In [3]:
# -*- coding: utf-8 -*-
"""
CSM triplet selection (min-dmid heuristic) + GBR regressor
---------------------------------------------------------
- Triplet selection: deterministic geometric "scorer":
    * Among all candidates per cluster, choose the one whose midpoint
      is closest to the DBSCAN cluster centroid (min dmid).
- Feature switch: angle/range (plus inten, snr, noise per radar).
- Localization model: GradientBoostingRegressor (squared_error), wrapped
  with MultiOutputRegressor for (x, y).
- Hyperparameter optimisation:
    * Small manual grid search over GBR hyperparameters using train/val split.
    * Selection metric: XY-RMSE on validation set.
- Saves:
    * Selected training triplets (CSV).
    * Train/val metrics for best GBR (CSV & bar plots).
    * Full hyperparameter grid results (CSV).
    * Best params (TXT).
    * Test metrics (CSV; incl. cc2-only).
    * Test scatter + error histogram for each feature_mode.
"""

import os, glob, math, random
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import DBSCAN
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor

import torch

plt.rcParams.update({
    "figure.dpi": 120, "axes.grid": True, "grid.linestyle": "--",
    "pdf.fonttype": 42, "ps.fonttype": 42
})

# ======================
# === Configuration  ===
# ======================

BASE_PATH = "/home/charithag/master thesis/practise/18Oct/corrected"

# Sessions
TRAIN_SUFFIXES = [
    "rr1","rr2","rr3","rr4","rr5","rr6","rr7","rr8","rr9","rr12","rr11",
    "cc3","cc5","cc6","cc7","cc8","pcc3","prr1","prr2","prr3","prr4",
    "pcc1","pcc2","cc11","cc12","cc13","cc10","cc1","cc2"
]
TEST_SUFFIXES  = ["cc2"]  # explicit cc2 test plots

# Radar roles
FUSION_RADARS   = ["rpi4","rpi1","rpi3"]
REFERENCE_RADARS= ["radar1","rpi2"]  # kept for completeness (not used here)

# Anchors + yaws (deg CCW+)
radar_positions: Dict[str, Tuple[float, float]] = {
    "rpi4": (0.0, 0.0),
    "rpi1": (0.07, 0.7),
    "rpi3": (-0.04, -0.75),
    "rpi5": (2.3, 0.4),
    "rpi6": (2.8, 1.7),
}
sides = {r: {"angle": 0.0} for r in radar_positions.keys()}

# File column mapping (ti_mmwave-style, 0-based)
CSV_X_COL        = 3
CSV_Y_COL        = 4
CSV_RANGE_COL    = 6
CSV_DOPPLER_COL  = 8
CSV_ANGLE_COL    = 9          # (1-based col10)
CSV_INTEN_COL    = 10         # (1-based col11)
CSV_SNR_COL      = 11         # (1-based col12)
CSV_NOISE_COL    = 12         # (1-based col13)
CSV_TIME_COL     = -1

# Pipeline knobs
BIN_SECONDS      = 0.20
DBSCAN_EPS       = 0.30
DBSCAN_MINPTS    = 3
TIME_TOLERANCE_S = 0.3
SPREAD_MAX_M     = 0.25
K_NEAREST_PER_RADAR = 3
REMOVE_DOPPLER_EQ: Optional[float] = 8.0

# Training knobs (GBR)
SEED        = 1337
VAL_SPLIT   = 0.15

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Output
OUT_DIR = "ML results"  # (user requested this exact folder name)

# ======================
# === Utilities      ===
# ======================

def ensure_dir(p): os.makedirs(p, exist_ok=True)
def savefig(path): ensure_dir(os.path.dirname(path)); plt.savefig(path, dpi=300, bbox_inches="tight"); plt.close()

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def _has_header(path: str) -> bool:
    try:
        with open(path,"r",errors="ignore") as f:
            toks = (f.readline().strip()).split(",")
        for t in toks:
            try: float(t)
            except ValueError: return True
        return False
    except: return False

def _read_any(path: str, header: Optional[int]) -> pd.DataFrame:
    try:    return pd.read_csv(path, header=header, sep=None, engine="python")
    except: return pd.read_csv(path, header=header, delim_whitespace=True)

def _get(df, spec):
    if isinstance(spec,int):
        idx = spec if spec>=0 else df.shape[1]+spec
        return df.iloc[:,idx]
    if isinstance(spec,str):
        if spec in df.columns: return df[spec]
        low = {c.lower():c for c in df.columns}
        if spec.lower() in low: return df[low[spec.lower()]]
    raise KeyError(spec)

def discover_files_for_radar(base: str, radar: str, suffix: str)->List[str]:
    exact = os.path.join(base, radar, suffix)
    if os.path.isfile(exact): return [exact]
    if os.path.isdir(exact):
        files = glob.glob(os.path.join(exact,"**","*"), recursive=True)
        return sorted([f for f in files if os.path.isfile(f)])
    root = os.path.join(base, radar)
    if os.path.isdir(root):
        candidates = glob.glob(os.path.join(root,"**","*"), recursive=True)
        hits = [f for f in candidates if os.path.isfile(f) and os.path.basename(f)==suffix]
        return sorted(hits)
    return []

def load_one_file(path: str) -> pd.DataFrame:
    header = 0 if _has_header(path) else None
    df = _read_any(path, header)
    out = pd.DataFrame({
        "x":        pd.to_numeric(_get(df, CSV_X_COL), errors="coerce"),
        "y":        pd.to_numeric(_get(df, CSV_Y_COL), errors="coerce"),
        "range":    pd.to_numeric(_get(df, CSV_RANGE_COL), errors="coerce"),
        "angle":    pd.to_numeric(_get(df, CSV_ANGLE_COL), errors="coerce"),
        "inten":    pd.to_numeric(_get(df, CSV_INTEN_COL), errors="coerce"),
        "snr":      pd.to_numeric(_get(df, CSV_SNR_COL), errors="coerce"),
        "noise":    pd.to_numeric(_get(df, CSV_NOISE_COL), errors="coerce"),
    })
    try: out["doppler"] = pd.to_numeric(_get(df, CSV_DOPPLER_COL), errors="coerce")
    except: out["doppler"] = np.nan
    try: out["timestamp"] = _get(df, CSV_TIME_COL)
    except: out["timestamp"] = np.nan
    return out.dropna(subset=["x","y"]).reset_index(drop=True)

def load_radar_suffix(base: str, radar: str, suffix: str) -> pd.DataFrame:
    files = discover_files_for_radar(base, radar, suffix)
    if not files:
        return pd.DataFrame(columns=["x","y","range","angle","inten","snr","noise","timestamp","radar"])
    parts=[]
    for f in files:
        try:
            df = load_one_file(f)
            if REMOVE_DOPPLER_EQ is not None and "doppler" in df.columns:
                df = df[~np.isclose(df["doppler"].astype(float), REMOVE_DOPPLER_EQ)]
            df["radar"]=radar
            parts.append(df)
        except Exception as e:
            print(f"[warn] load {f}: {e}")
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

def rot2d(theta):
    c,s=math.cos(theta), math.sin(theta)
    return np.array([[c,-s],[s,c]],float)

def transform_local_to_world(df: pd.DataFrame, yaw_deg: float, tx: float, ty: float)->pd.DataFrame:
    R = rot2d(math.radians(yaw_deg))
    xy = df[["x","y"]].to_numpy(float) @ R.T
    out = df.copy()
    out["xw"]=xy[:,0]+tx; out["yw"]=xy[:,1]+ty
    return out

def load_transform_suffix_all_radars(sfx: str) -> pd.DataFrame:
    parts=[]
    for r,(tx,ty) in radar_positions.items():
        d = load_radar_suffix(BASE_PATH, r, sfx)
        if d.empty: continue
        yaw = float(sides.get(r,{}).get("angle",0.0))
        d = transform_local_to_world(d, yaw, tx, ty)
        parts.append(d)
    if not parts: return pd.DataFrame()
    df = pd.concat(parts, ignore_index=True)
    return df[["x","y","range","angle","inten","snr","noise","timestamp","radar","xw","yw"]]

def to_seconds(s: pd.Series)->pd.Series:
    a = s.astype(str).str.extract(r'([0-9]+(?:\.[0-9]+)?)')[0]
    return pd.to_numeric(a, errors="coerce")

# =============== Binning + Clustering ===============

def bin_time(df: pd.DataFrame, bin_s: float)->pd.DataFrame:
    df = df.dropna(subset=["timestamp"]).copy()
    df["t"] = to_seconds(df["timestamp"])
    df = df.dropna(subset=["t"])
    df["tbin"] = np.floor(df["t"]/bin_s).astype(int)
    return df

def cluster_bin(df_bin: pd.DataFrame)->pd.DataFrame:
    if df_bin.empty: return pd.DataFrame()
    X = df_bin[["xw","yw"]].to_numpy(float)
    labels = DBSCAN(eps=DBSCAN_EPS, min_samples=DBSCAN_MINPTS).fit_predict(X)
    out = df_bin.copy()
    out["cluster"]=labels
    return out

def compute_centroids(df_bin: pd.DataFrame)->Dict[int, Tuple[float,float]]:
    cents={}
    for c, d in df_bin.groupby("cluster"):
        if c==-1: continue
        cents[c]=(float(d["xw"].mean()), float(d["yw"].mean()))
    return cents

# =============== Candidate triplets per cluster ===============

def nearest_k_to_centroid(d: pd.DataFrame, centroid: Tuple[float,float], k:int)->pd.DataFrame:
    dx = d["xw"].to_numpy()-centroid[0]
    dy = d["yw"].to_numpy()-centroid[1]
    dists = np.hypot(dx,dy)
    idx = np.argsort(dists)[:min(k,len(d))]
    return d.iloc[idx].copy()

def build_candidates_for_cluster(d_cluster: pd.DataFrame,
                                 centroid: Tuple[float,float],
                                 fusion_radars: List[str],
                                 k_each:int=K_NEAREST_PER_RADAR)->List[dict]:
    per_radar={}
    for r in fusion_radars:
        dr = d_cluster[d_cluster["radar"]==r]
        if dr.empty:
            return []  # must have all fusion radars
        kn = nearest_k_to_centroid(dr, centroid, k_each).reset_index(drop=True)
        per_radar[r]=kn

    R1,R2,R3 = fusion_radars[:3]
    cands=[]
    n1, n2, n3 = len(per_radar[R1]), len(per_radar[R2]), len(per_radar[R3])
    for ia in range(n1):
        a = per_radar[R1].iloc[ia]
        for ib in range(n2):
            b = per_radar[R2].iloc[ib]
            for ic in range(n3):
                c = per_radar[R3].iloc[ic]
                # time gate
                t_ok = (abs(a["t"]-b["t"])<=TIME_TOLERANCE_S and
                        abs(a["t"]-c["t"])<=TIME_TOLERANCE_S)
                if not t_ok:
                    continue
                # spread gate
                P = np.array([[a["xw"],a["yw"]],
                              [b["xw"],b["yw"]],
                              [c["xw"],c["yw"]]], float)
                d01 = np.hypot(*(P[0]-P[1])); d02 = np.hypot(*(P[0]-P[2])); d12 = np.hypot(*(P[1]-P[2]))
                spread = max(d01,d02,d12)
                if spread>SPREAD_MAX_M:
                    continue
                cands.append({
                    "rows": (ia, ib, ic),
                    "points": P,                 # [3,2] world coords
                    "times": (float(a["t"]), float(b["t"]), float(c["t"])),
                    "centroid": (float(P[:,0].mean()), float(P[:,1].mean()))
                })
    return cands

# =============== Triplet features (for min-dmid) ===============

def triplet_features_for_scoring(cand: dict, cluster_centroid: Tuple[float,float])->np.ndarray:
    """
    z = [d01, d02, d12, spread, dmid, |t0-t1|, |t0-t2|, |t1-t2|]
    We use index 4 (dmid) to implement the min-dmid selector.
    """
    P = cand["points"]; t0,t1,t2 = cand["times"]
    d01 = np.hypot(*(P[0]-P[1])); d02 = np.hypot(*(P[0]-P[2])); d12 = np.hypot(*(P[1]-P[2]))
    spread = max(d01,d02,d12)
    mid = P.mean(axis=0)
    dmid = np.hypot(*(mid - np.array(cluster_centroid)))
    dt = np.array([abs(t0-t1), abs(t0-t2), abs(t1-t2)], float)
    z = np.array([d01,d02,d12,spread,dmid, dt[0],dt[1],dt[2]], float)
    return z

# =============== Selection using min-dmid heuristic ===============

def select_triplets_for_suffix(sfx: str)->pd.DataFrame:
    """
    Deterministic triplet selection for a suffix:
    - Time-bin + DBSCAN per bin.
    - For each cluster:
        * Build candidate triplets (K-nearest per radar, time/spread gates).
        * For each candidate, compute features z.
        * Select candidate with minimal dmid (midpoint-to-cluster-centroid distance).
    """
    df = load_transform_suffix_all_radars(sfx)
    if df.empty: return pd.DataFrame()
    df = bin_time(df, BIN_SECONDS)
    rows=[]
    for tbin, d_bin in df.groupby("tbin"):
        dc = cluster_bin(d_bin)
        if dc.empty:
            continue
        cents = compute_centroids(dc)
        if not cents: continue
        for cid, d_cluster in dc.groupby("cluster"):
            if cid==-1: continue
            centroid = cents[cid]
            cands = build_candidates_for_cluster(d_cluster, centroid, FUSION_RADARS, K_NEAREST_PER_RADAR)
            if len(cands)==0:
                continue

            # --- min-dmid selection ---
            dmid_values = []
            for c in cands:
                z = triplet_features_for_scoring(c, centroid)
                dmid_values.append(z[4])  # index 4 = dmid
            best_idx = int(np.argmin(dmid_values))
            best = cands[best_idx]
            # ----------------------------

            # rebuild per_radar table to pick actual rows by index
            per_radar={}
            for r in FUSION_RADARS:
                dr = d_cluster[d_cluster["radar"]==r]
                if dr.empty:
                    per_radar = {}
                    break
                per_radar[r]=nearest_k_to_centroid(dr, centroid, K_NEAREST_PER_RADAR).reset_index(drop=True)
            if len(per_radar) < 3:
                continue

            idx1, idx2, idx3 = best["rows"]
            sel = [per_radar[FUSION_RADARS[0]].iloc[idx1],
                   per_radar[FUSION_RADARS[1]].iloc[idx2],
                   per_radar[FUSION_RADARS[2]].iloc[idx3]]

            points = best["points"]; times = best["times"]
            trip = {"suffix": sfx, "tbin": int(tbin), "cluster": int(cid),
                    "x_avg": float(points[:,0].mean()), "y_avg": float(points[:,1].mean()),
                    "t_ref": float(np.mean(times))}
            for r, srow in zip(FUSION_RADARS, sel):
                for k in ["angle","range","inten","snr","noise"]:
                    trip[f"{r}_{k}"]=float(srow[k])
            rows.append(trip)
    return pd.DataFrame(rows)

# =============== Localization model (GBR) ===============

def build_Xy_from_triplets(df_sel: pd.DataFrame, feature_mode:str)->Tuple[np.ndarray,np.ndarray,List[str]]:
    if df_sel.empty: return np.empty((0,12)), np.empty((0,2)), []
    feat_cols=[]
    for r in FUSION_RADARS:
        if feature_mode=="angle":
            feat_cols += [f"{r}_angle", f"{r}_inten", f"{r}_snr", f"{r}_noise"]
        elif feature_mode=="range":
            feat_cols += [f"{r}_range", f"{r}_inten", f"{r}_snr", f"{r}_noise"]
        else:
            raise ValueError("FEATURE_MODE must be 'angle' or 'range'")
    X = df_sel[feat_cols].to_numpy(float)
    y = df_sel[["x_avg","y_avg"]].to_numpy(float)
    return X,y,feat_cols

def metrics_from_df(df: pd.DataFrame)->dict:
    dx = (df["x_pred"]-df["x_true"]).to_numpy(float)
    dy = (df["y_pred"]-df["y_true"]).to_numpy(float)
    err = np.hypot(dx,dy)
    return {
        "x_rmse": float(np.sqrt(np.mean(dx**2))),
        "y_rmse": float(np.sqrt(np.mean(dy**2))),
        "xy_rmse": float(np.sqrt(np.mean(err**2))),
        "xy_mae": float(np.mean(err)),
        "p90": float(np.percentile(err,90)),
        "N": int(len(df))
    }

def scatter_plot(y_true, y_pred, title, path):
    plt.figure(figsize=(5,5))
    plt.scatter(y_true[:,0], y_true[:,1], s=10, label="true")
    plt.scatter(y_pred[:,0], y_pred[:,1], s=10, label="pred")
    plt.axis("equal"); plt.title(title); plt.legend(); plt.tight_layout()
    savefig(path)

def hist_plot(y_true, y_pred, title, path):
    err=np.hypot(y_pred[:,0]-y_true[:,0], y_pred[:,1]-y_true[:,1])
    plt.figure(figsize=(6,4))
    plt.hist(err, bins=40, alpha=0.9)
    plt.xlabel("||error|| [m]"); plt.title(title); plt.tight_layout()
    savefig(path)

def eval_model_on_suffixes_gbr(gbr_predictor, X_builder, suffixes, feature_mode, tag, out_subdir):
    all_rows=[]
    for sfx in suffixes:
        sel = select_triplets_for_suffix(sfx)
        if sel.empty:
            print(f"[test {sfx}] no selected triplets")
            continue
        X_raw, y, feat_cols = X_builder(sel, feature_mode)
        Xs = gbr_predictor["prep"](X_raw)  # preprocessed X
        y_pred = gbr_predictor["pred"](Xs) # np array [N,2]

        df = pd.DataFrame({
            "suffix": sfx,
            "x_true": y[:,0], "y_true": y[:,1],
            "x_pred": y_pred[:,0], "y_pred": y_pred[:,1],
            "tbin": sel["tbin"].values, "cluster": sel["cluster"].values
        })
        all_rows.append(df)

        # save predictions CSV per suffix
        df.to_csv(os.path.join(out_subdir, f"{tag}_{feature_mode}_GBR_{sfx}_predictions.csv"), index=False)

        # plots: scatter + hist
        scatter_plot(y, y_pred,
                     f"Test {sfx} — GBR ({feature_mode})",
                     os.path.join(out_subdir, f"{tag}_{feature_mode}_GBR_{sfx}_test_scatter.png"))
        hist_plot(y, y_pred,
                  f"Test {sfx} — GBR ({feature_mode}) error",
                  os.path.join(out_subdir, f"{tag}_{feature_mode}_GBR_{sfx}_test_errhist.png"))

    return pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()

# =============== Hyperparameter optimisation ===============

def tune_gbr_hyperparams(Xtr_raw: np.ndarray,
                         ytr: np.ndarray,
                         Xval_raw: np.ndarray,
                         yval: np.ndarray,
                         tag_base: str,
                         feature_mode: str,
                         out_subdir: str) -> dict:
    """
    Simple manual grid search for GBR hyperparameters using train/val split.
    Selection metric: XY-RMSE on validation.
    Returns a dict with:
        "params"         - best hyperparameter dict
        "imp"            - fitted imputer
        "sca"            - fitted scaler
        "model"          - fitted MultiOutputRegressor(GBR)
        "train_metrics"  - metrics on train
        "val_metrics"    - metrics on val
    Also saves a CSV with all tried configs + their metrics.
    """
    # Grid can be extended if you want more exhaustive search
    param_grid = [
        {"n_estimators": 200, "learning_rate": 0.05, "max_depth": 2, "subsample": 1},
        {"n_estimators": 300, "learning_rate": 0.05, "max_depth": 3, "subsample": 1},
        {"n_estimators": 400, "learning_rate": 0.10, "max_depth": 3, "subsample": 1},
        {"n_estimators": 600, "learning_rate": 0.05, "max_depth": 4, "subsample": 1},
        {"n_estimators": 600, "learning_rate": 0.05, "max_depth": 3, "subsample": 1},
        
        {"n_estimators": 600, "learning_rate": 0.02, "max_depth": 4, "subsample": 1},
        {"n_estimators": 800, "learning_rate": 0.05, "max_depth": 4, "subsample": 1},
        {"n_estimators": 1000, "learning_rate": 0.02, "max_depth": 4, "subsample": 1},
        {"n_estimators": 1200, "learning_rate": 0.02, "max_depth": 4, "subsample": 1},
        {"n_estimators": 1500, "learning_rate": 0.02, "max_depth": 4, "subsample": 1},
        {"n_estimators": 1500, "learning_rate": 0.05, "max_depth": 4, "subsample": 1},
        {"n_estimators": 1500, "learning_rate": 0.05, "max_depth": 3, "subsample": 1},
        {"n_estimators": 1500, "learning_rate": 0.05, "max_depth": 5, "subsample": 1},
        
        {"n_estimators": 2000, "learning_rate": 0.02, "max_depth": 4, "subsample": 1},
        {"n_estimators": 2500, "learning_rate": 0.02, "max_depth": 4, "subsample": 1},

    ]

    results = []
    best_cfg = None
    best_val_rmse = float("inf")

    for i, params in enumerate(param_grid):
        print(f"[GBR tune {feature_mode}] config {i+1}/{len(param_grid)}: {params}")

        imp = SimpleImputer(strategy="mean")
        sca = StandardScaler()

        Xtr0 = imp.fit_transform(Xtr_raw)
        Xval0 = imp.transform(Xval_raw)
        Xtr = sca.fit_transform(Xtr0)
        Xval = sca.transform(Xval0)

        gbr_base = GradientBoostingRegressor(
            n_estimators=params["n_estimators"],
            learning_rate=params["learning_rate"],
            max_depth=params["max_depth"],
            subsample=params["subsample"],
            loss="squared_error",
            random_state=SEED
        )
        gbr = MultiOutputRegressor(gbr_base)
        gbr.fit(Xtr, ytr)

        # Train metrics
        ytr_pred = gbr.predict(Xtr)
        df_tr = pd.DataFrame({
            "x_true": ytr[:,0], "y_true": ytr[:,1],
            "x_pred": ytr_pred[:,0], "y_pred": ytr_pred[:,1]
        })
        m_tr = metrics_from_df(df_tr)

        # Val metrics
        yval_pred = gbr.predict(Xval)
        df_va = pd.DataFrame({
            "x_true": yval[:,0], "y_true": yval[:,1],
            "x_pred": yval_pred[:,0], "y_pred": yval_pred[:,1]
        })
        m_va = metrics_from_df(df_va)

        row = {
            "n_estimators": params["n_estimators"],
            "learning_rate": params["learning_rate"],
            "max_depth": params["max_depth"],
            "subsample": params["subsample"],
            "train_xy_rmse": m_tr["xy_rmse"],
            "train_xy_mae":  m_tr["xy_mae"],
            "val_xy_rmse":   m_va["xy_rmse"],
            "val_xy_mae":    m_va["xy_mae"],
        }
        results.append(row)

        if m_va["xy_rmse"] < best_val_rmse:
            best_val_rmse = m_va["xy_rmse"]
            best_cfg = {
                "params": params,
                "imp": imp,
                "sca": sca,
                "model": gbr,
                "train_metrics": m_tr,
                "val_metrics": m_va,
            }

    # Save grid search results
    df_grid = pd.DataFrame(results)
    df_grid.to_csv(os.path.join(
        out_subdir,
        f"{tag_base}_{feature_mode}_GBR_hyperparam_grid.csv"
    ), index=False)

    print(f"[GBR tune {feature_mode}] best val XY-RMSE={best_val_rmse:.3f} with params={best_cfg['params']}")

    # Save best params as a small text file
    best_txt_path = os.path.join(
        out_subdir,
        f"{tag_base}_{feature_mode}_GBR_best_params.txt"
    )
    with open(best_txt_path, "w") as f:
        f.write(f"Best params (feature_mode={feature_mode}):\n")
        for k,v in best_cfg["params"].items():
            f.write(f"{k}: {v}\n")
        f.write(f"\nBest val XY-RMSE: {best_val_rmse:.6f}\n")

    return best_cfg

# =============== Runner for one feature_mode ===============

def run_gbr_for_feature_mode(train_sel: pd.DataFrame, feature_mode: str, tag_base: str):
    """
    Train/eval GBR (squared error) for one feature_mode.
    Steps:
      - Build X,y from selected triplets.
      - Train/val split.
      - Hyperparameter tuning (manual grid search) using val XY-RMSE.
      - Save:
          * hyperparameter grid results
          * best-GBR train/val metrics CSV + bar plot
          * best hyperparameters (TXT)
          * test metrics CSV (all + cc2-only)
          * predictions & plots for TEST_SUFFIXES
    """
    ensure_dir(OUT_DIR)
    out_subdir = os.path.join(OUT_DIR, f"{tag_base}_{feature_mode}_GBR")
    ensure_dir(out_subdir)

    # === Build X,y
    X_raw, y, feat_cols = build_Xy_from_triplets(train_sel, feature_mode)
    if len(X_raw)==0:
        raise RuntimeError(f"No training rows for feature_mode={feature_mode}")

    # shared train/val split
    Xtr_raw, Xval_raw, ytr, yval = train_test_split(X_raw, y, test_size=VAL_SPLIT, random_state=SEED)

    # === Hyperparameter optimisation for GBR
    best_cfg = tune_gbr_hyperparams(
        Xtr_raw, ytr, Xval_raw, yval,
        tag_base=tag_base,
        feature_mode=feature_mode,
        out_subdir=out_subdir
    )

    imp_gb = best_cfg["imp"]
    sca_gb = best_cfg["sca"]
    gbr     = best_cfg["model"]
    m_tr    = best_cfg["train_metrics"]
    m_va    = best_cfg["val_metrics"]

    def gbr_prep(X):
        X0 = imp_gb.transform(X)
        return sca_gb.transform(X0)

    gbr_predictor = {
        "name": "GBR",
        "prep": gbr_prep,
        "pred": lambda Xs: gbr.predict(Xs)
    }

    # === Train/val metrics (from best config)
    ytr_pred = gbr_predictor["pred"](gbr_predictor["prep"](Xtr_raw))
    yval_pred = gbr_predictor["pred"](gbr_predictor["prep"](Xval_raw))

    df_tr = pd.DataFrame({
        "x_true": ytr[:,0], "y_true": ytr[:,1],
        "x_pred": ytr_pred[:,0], "y_pred": ytr_pred[:,1]
    })
    df_va = pd.DataFrame({
        "x_true": yval[:,0], "y_true": yval[:,1],
        "x_pred": yval_pred[:,0], "y_pred": yval_pred[:,1]
    })

    # re-compute metrics for consistency (or use m_tr/m_va directly)
    m_tr = metrics_from_df(df_tr)
    m_va = metrics_from_df(df_va)

    df_tv = pd.DataFrame([
        {"subset": "train", **m_tr},
        {"subset": "val",   **m_va}
    ])
    df_tv.to_csv(os.path.join(out_subdir,
        f"{tag_base}_{feature_mode}_GBR_train_val_metrics.csv"), index=False)

    # bar plot: train vs val XY RMSE & MAE
    rmse_vals = [m_tr["xy_rmse"], m_va["xy_rmse"]]
    mae_vals  = [m_tr["xy_mae"],  m_va["xy_mae"]]
    labels    = ["train", "val"]

    x = np.arange(len(labels))
    width = 0.35

    plt.figure(figsize=(6,4))
    plt.bar(x - width/2, rmse_vals, width, label="XY RMSE")
    plt.bar(x + width/2, mae_vals,  width, label="XY MAE")
    plt.xticks(x, labels)
    plt.ylabel("Error [m]")
    plt.title(f"{tag_base} ({feature_mode}) — GBR train/val metrics (tuned)")
    plt.legend()
    plt.tight_layout()
    savefig(os.path.join(out_subdir,
        f"{tag_base}_{feature_mode}_GBR_train_val_bar.png"))

    # === Evaluate on TEST suffixes with the tuned model
    df_test = eval_model_on_suffixes_gbr(gbr_predictor, build_Xy_from_triplets,
                                         TEST_SUFFIXES, feature_mode,
                                         tag=tag_base, out_subdir=out_subdir)

    metrics_rows=[]
    if not df_test.empty:
        m_all = metrics_from_df(df_test)
        metrics_rows.append({"feature_mode":feature_mode, "model":"GBR", **m_all})
        df_cc2 = df_test[df_test["suffix"]=="cc2"]
        if not df_cc2.empty:
            m_cc2 = metrics_from_df(df_cc2)
            metrics_rows.append({"feature_mode":feature_mode, "model":"GBR_cc2_only", **m_cc2})

    if metrics_rows:
        df_metrics = pd.DataFrame(metrics_rows)
        df_metrics.to_csv(os.path.join(out_subdir,
            f"{tag_base}_{feature_mode}_GBR_test_metrics.csv"), index=False)

# =============== Main ===============

def main():
    set_seed(SEED)
    ensure_dir(OUT_DIR)
    print(f"[config] device={DEVICE}")
    print("[info] Triplet selector = min-dmid (geometric heuristic)")
    print("[info] Localization model = GBR (squared_error) with hyperparameter tuning")

    # 1) Build selected triplets for TRAIN using min-dmid
    sel_rows=[]
    for sfx in TRAIN_SUFFIXES:
        sel = select_triplets_for_suffix(sfx)
        if sel.empty:
            print(f"[train sel] {sfx}: 0 rows"); continue
        sel_rows.append(sel)
        print(f"[train sel] {sfx}: {len(sel)} rows")
    if not sel_rows:
        print("[abort] no selected triplets on TRAIN."); return
    train_sel = pd.concat(sel_rows, ignore_index=True)
    train_sel.to_csv(os.path.join(OUT_DIR, "train_selected_triplets_min_dmid.csv"), index=False)

    # 2) Run GBR for BOTH feature modes (angle & range)
    for FEATURE_MODE in ["angle","range"]:
        print(f"\n=== Running GBR for FEATURE_MODE={FEATURE_MODE} ===")
        run_gbr_for_feature_mode(train_sel, FEATURE_MODE, tag_base="CSM")

if __name__=="__main__":
    main()


[config] device=cuda
[info] Triplet selector = min-dmid (geometric heuristic)
[info] Localization model = GBR (squared_error) with hyperparameter tuning
[train sel] rr1: 46 rows
[train sel] rr2: 34 rows
[train sel] rr3: 53 rows
[train sel] rr4: 35 rows
[train sel] rr5: 39 rows
[train sel] rr6: 32 rows
[train sel] rr7: 31 rows
[train sel] rr8: 49 rows
[train sel] rr9: 55 rows
[train sel] rr12: 41 rows
[train sel] rr11: 38 rows
[train sel] cc3: 37 rows
[train sel] cc5: 52 rows
[train sel] cc6: 38 rows
[train sel] cc7: 35 rows
[train sel] cc8: 42 rows
[train sel] pcc3: 95 rows
[train sel] prr1: 39 rows
[train sel] prr2: 53 rows
[train sel] prr3: 97 rows
[train sel] prr4: 0 rows
[train sel] pcc1: 80 rows
[train sel] pcc2: 79 rows
[train sel] cc11: 56 rows
[train sel] cc12: 36 rows
[train sel] cc13: 39 rows
[train sel] cc10: 53 rows
[train sel] cc1: 49 rows
[train sel] cc2: 54 rows

=== Running GBR for FEATURE_MODE=angle ===
[GBR tune angle] config 1/15: {'n_estimators': 200, 'learning_rate

In [8]:
# -*- coding: utf-8 -*-
"""
CSM triplet selection (min-dmid heuristic) + GBR regressor
---------------------------------------------------------
- Triplet selection: deterministic geometric "scorer":
    * Among all candidates per cluster, choose the one whose midpoint
      is closest to the DBSCAN cluster centroid (min dmid).
- Feature switch: angle/range (plus inten, snr, noise per radar).
- Localization model: GradientBoostingRegressor (squared_error), wrapped
  with MultiOutputRegressor for (x, y).
- Hyperparameter optimisation:
    * CSM geometry (DBSCAN_EPS, DBSCAN_MINPTS, SPREAD_MAX_M, TIME_TOLERANCE_S)
      via angle-mode + fixed GBR baseline.
    * GBR hyperparameters via manual grid search per feature_mode (angle/range).
- Saves:
    * Selected training triplets (CSV) for final tuned CSM.
    * CSM geometry grid + best config (CSV/TXT).
    * Train/val metrics for best GBR (CSV & bar plots).
    * GBR hyperparameter grid per feature_mode (CSV).
    * Test metrics (CSV; incl. cc2-only).
    * Test scatter + error histogram for each feature_mode.
"""

import os, glob, math, random
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import DBSCAN
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor

import torch

plt.rcParams.update({
    "figure.dpi": 120, "axes.grid": True, "grid.linestyle": "--",
    "pdf.fonttype": 42, "ps.fonttype": 42
})

# ======================
# === Configuration  ===
# ======================

BASE_PATH = "/home/charithag/master thesis/practise/18Oct/corrected"

# Sessions
TRAIN_SUFFIXES = [
    "rr1","rr2","rr3","rr4","rr5","rr6","rr7","rr8","rr9","rr12","rr11",
    "cc3","cc5","cc6","cc7","cc8","pcc3","prr1","prr2","prr3","prr4",
    "pcc1","pcc2","cc11","cc12","cc13","cc10","cc1","cc2"
]
TEST_SUFFIXES  = ["cc2"]  # explicit cc2 test plots

# Radar roles
FUSION_RADARS   = ["rpi4","rpi1","rpi3"]
REFERENCE_RADARS= ["radar1","rpi2"]  # kept for completeness (not used here)

# Anchors + yaws (deg CCW+)
radar_positions: Dict[str, Tuple[float, float]] = {
    "rpi4": (0.0, 0.0),
    "rpi1": (0.07, 0.7),
    "rpi3": (-0.04, -0.75),
    "rpi5": (2.3, 0.4),
    "rpi6": (2.8, 1.7),
}
sides = {r: {"angle": 0.0} for r in radar_positions.keys()}

# File column mapping (ti_mmwave-style, 0-based)
CSV_X_COL        = 3
CSV_Y_COL        = 4
CSV_RANGE_COL    = 6
CSV_DOPPLER_COL  = 8
CSV_ANGLE_COL    = 9          # (1-based col10)
CSV_INTEN_COL    = 10         # (1-based col11)
CSV_SNR_COL      = 11         # (1-based col12)
CSV_NOISE_COL    = 12         # (1-based col13)
CSV_TIME_COL     = -1

# Pipeline knobs (initial defaults; may be tuned)
BIN_SECONDS      = 0.20
DBSCAN_EPS       = 0.30
DBSCAN_MINPTS    = 3
TIME_TOLERANCE_S = 0.5
SPREAD_MAX_M     = 0.45
K_NEAREST_PER_RADAR = 3
REMOVE_DOPPLER_EQ: Optional[float] = 8.0

# Training knobs (GBR)
SEED        = 1337
VAL_SPLIT   = 0.15

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Output
OUT_DIR = "ML results"  # (user requested this exact folder name)

# ======================
# === Utilities      ===
# ======================

def ensure_dir(p): os.makedirs(p, exist_ok=True)
def savefig(path): ensure_dir(os.path.dirname(path)); plt.savefig(path, dpi=300, bbox_inches="tight"); plt.close()

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def _has_header(path: str) -> bool:
    try:
        with open(path,"r",errors="ignore") as f:
            toks = (f.readline().strip()).split(",")
        for t in toks:
            try: float(t)
            except ValueError: return True
        return False
    except: return False

def _read_any(path: str, header: Optional[int]) -> pd.DataFrame:
    try:    return pd.read_csv(path, header=header, sep=None, engine="python")
    except: return pd.read_csv(path, header=header, delim_whitespace=True)

def _get(df, spec):
    if isinstance(spec,int):
        idx = spec if spec>=0 else df.shape[1]+spec
        return df.iloc[:,idx]
    if isinstance(spec,str):
        if spec in df.columns: return df[spec]
        low = {c.lower():c for c in df.columns}
        if spec.lower() in low: return df[low[spec.lower()]]
    raise KeyError(spec)

def discover_files_for_radar(base: str, radar: str, suffix: str)->List[str]:
    exact = os.path.join(base, radar, suffix)
    if os.path.isfile(exact): return [exact]
    if os.path.isdir(exact):
        files = glob.glob(os.path.join(exact,"**","*"), recursive=True)
        return sorted([f for f in files if os.path.isfile(f)])
    root = os.path.join(base, radar)
    if os.path.isdir(root):
        candidates = glob.glob(os.path.join(root,"**","*"), recursive=True)
        hits = [f for f in candidates if os.path.isfile(f) and os.path.basename(f)==suffix]
        return sorted(hits)
    return []

def load_one_file(path: str) -> pd.DataFrame:
    header = 0 if _has_header(path) else None
    df = _read_any(path, header)
    out = pd.DataFrame({
        "x":        pd.to_numeric(_get(df, CSV_X_COL), errors="coerce"),
        "y":        pd.to_numeric(_get(df, CSV_Y_COL), errors="coerce"),
        "range":    pd.to_numeric(_get(df, CSV_RANGE_COL), errors="coerce"),
        "angle":    pd.to_numeric(_get(df, CSV_ANGLE_COL), errors="coerce"),
        "inten":    pd.to_numeric(_get(df, CSV_INTEN_COL), errors="coerce"),
        "snr":      pd.to_numeric(_get(df, CSV_SNR_COL), errors="coerce"),
        "noise":    pd.to_numeric(_get(df, CSV_NOISE_COL), errors="coerce"),
    })
    try: out["doppler"] = pd.to_numeric(_get(df, CSV_DOPPLER_COL), errors="coerce")
    except: out["doppler"] = np.nan
    try: out["timestamp"] = _get(df, CSV_TIME_COL)
    except: out["timestamp"] = np.nan
    return out.dropna(subset=["x","y"]).reset_index(drop=True)

def load_radar_suffix(base: str, radar: str, suffix: str) -> pd.DataFrame:
    files = discover_files_for_radar(base, radar, suffix)
    if not files:
        return pd.DataFrame(columns=["x","y","range","angle","inten","snr","noise","timestamp","radar"])
    parts=[]
    for f in files:
        try:
            df = load_one_file(f)
            if REMOVE_DOPPLER_EQ is not None and "doppler" in df.columns:
                df = df[~np.isclose(df["doppler"].astype(float), REMOVE_DOPPLER_EQ)]
            df["radar"]=radar
            parts.append(df)
        except Exception as e:
            print(f"[warn] load {f}: {e}")
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

def rot2d(theta):
    c,s=math.cos(theta), math.sin(theta)
    return np.array([[c,-s],[s,c]],float)

def transform_local_to_world(df: pd.DataFrame, yaw_deg: float, tx: float, ty: float)->pd.DataFrame:
    R = rot2d(math.radians(yaw_deg))
    xy = df[["x","y"]].to_numpy(float) @ R.T
    out = df.copy()
    out["xw"]=xy[:,0]+tx; out["yw"]=xy[:,1]+ty
    return out

def load_transform_suffix_all_radars(sfx: str) -> pd.DataFrame:
    parts=[]
    for r,(tx,ty) in radar_positions.items():
        d = load_radar_suffix(BASE_PATH, r, sfx)
        if d.empty: continue
        yaw = float(sides.get(r,{}).get("angle",0.0))
        d = transform_local_to_world(d, yaw, tx, ty)
        parts.append(d)
    if not parts: return pd.DataFrame()
    df = pd.concat(parts, ignore_index=True)
    return df[["x","y","range","angle","inten","snr","noise","timestamp","radar","xw","yw"]]

def to_seconds(s: pd.Series)->pd.Series:
    a = s.astype(str).str.extract(r'([0-9]+(?:\.[0-9]+)?)')[0]
    return pd.to_numeric(a, errors="coerce")

# =============== Binning + Clustering ===============

def bin_time(df: pd.DataFrame, bin_s: float)->pd.DataFrame:
    df = df.dropna(subset=["timestamp"]).copy()
    df["t"] = to_seconds(df["timestamp"])
    df = df.dropna(subset=["t"])
    df["tbin"] = np.floor(df["t"]/bin_s).astype(int)
    return df

def cluster_bin(df_bin: pd.DataFrame)->pd.DataFrame:
    if df_bin.empty: return pd.DataFrame()
    X = df_bin[["xw","yw"]].to_numpy(float)
    labels = DBSCAN(eps=DBSCAN_EPS, min_samples=DBSCAN_MINPTS).fit_predict(X)
    out = df_bin.copy()
    out["cluster"]=labels
    return out

def compute_centroids(df_bin: pd.DataFrame)->Dict[int, Tuple[float,float]]:
    cents={}
    for c, d in df_bin.groupby("cluster"):
        if c==-1: continue
        cents[c]=(float(d["xw"].mean()), float(d["yw"].mean()))
    return cents

# =============== Candidate triplets per cluster ===============

def nearest_k_to_centroid(d: pd.DataFrame, centroid: Tuple[float,float], k:int)->pd.DataFrame:
    dx = d["xw"].to_numpy()-centroid[0]
    dy = d["yw"].to_numpy()-centroid[1]
    dists = np.hypot(dx,dy)
    idx = np.argsort(dists)[:min(k,len(d))]
    return d.iloc[idx].copy()

def build_candidates_for_cluster(d_cluster: pd.DataFrame,
                                 centroid: Tuple[float,float],
                                 fusion_radars: List[str],
                                 k_each:int=K_NEAREST_PER_RADAR)->List[dict]:
    per_radar={}
    for r in fusion_radars:
        dr = d_cluster[d_cluster["radar"]==r]
        if dr.empty:
            return []  # must have all fusion radars
        kn = nearest_k_to_centroid(dr, centroid, k_each).reset_index(drop=True)
        per_radar[r]=kn

    R1,R2,R3 = fusion_radars[:3]
    cands=[]
    n1, n2, n3 = len(per_radar[R1]), len(per_radar[R2]), len(per_radar[R3])
    for ia in range(n1):
        a = per_radar[R1].iloc[ia]
        for ib in range(n2):
            b = per_radar[R2].iloc[ib]
            for ic in range(n3):
                c = per_radar[R3].iloc[ic]
                # time gate
                t_ok = (abs(a["t"]-b["t"])<=TIME_TOLERANCE_S and
                        abs(a["t"]-c["t"])<=TIME_TOLERANCE_S)
                if not t_ok:
                    continue
                # spread gate
                P = np.array([[a["xw"],a["yw"]],
                              [b["xw"],b["yw"]],
                              [c["xw"],c["yw"]]], float)
                d01 = np.hypot(*(P[0]-P[1])); d02 = np.hypot(*(P[0]-P[2])); d12 = np.hypot(*(P[1]-P[2]))
                spread = max(d01,d02,d12)
                if spread>SPREAD_MAX_M:
                    continue
                cands.append({
                    "rows": (ia, ib, ic),
                    "points": P,                 # [3,2] world coords
                    "times": (float(a["t"]), float(b["t"]), float(c["t"])),
                    "centroid": (float(P[:,0].mean()), float(P[:,1].mean()))
                })
    return cands

# =============== Triplet features (for min-dmid) ===============

def triplet_features_for_scoring(cand: dict, cluster_centroid: Tuple[float,float])->np.ndarray:
    """
    z = [d01, d02, d12, spread, dmid, |t0-t1|, |t0-t2|, |t1-t2|]
    We use index 4 (dmid) to implement the min-dmid selector.
    """
    P = cand["points"]; t0,t1,t2 = cand["times"]
    d01 = np.hypot(*(P[0]-P[1])); d02 = np.hypot(*(P[0]-P[2])); d12 = np.hypot(*(P[1]-P[2]))
    spread = max(d01,d02,d12)
    mid = P.mean(axis=0)
    dmid = np.hypot(*(mid - np.array(cluster_centroid)))
    dt = np.array([abs(t0-t1), abs(t0-t2), abs(t1-t2)], float)
    z = np.array([d01,d02,d12,spread,dmid, dt[0],dt[1],dt[2]], float)
    return z

# =============== Selection using min-dmid heuristic ===============

def select_triplets_for_suffix(sfx: str)->pd.DataFrame:
    """
    Deterministic triplet selection for a suffix:
    - Time-bin + DBSCAN per bin.
    - For each cluster:
        * Build candidate triplets (K-nearest per radar, time/spread gates).
        * For each candidate, compute features z.
        * Select candidate with minimal dmid (midpoint-to-cluster-centroid distance).
    Uses global CSM geometry knobs: BIN_SECONDS, DBSCAN_EPS, DBSCAN_MINPTS,
    SPREAD_MAX_M, TIME_TOLERANCE_S, K_NEAREST_PER_RADAR.
    """
    df = load_transform_suffix_all_radars(sfx)
    if df.empty: return pd.DataFrame()
    df = bin_time(df, BIN_SECONDS)
    rows=[]
    for tbin, d_bin in df.groupby("tbin"):
        dc = cluster_bin(d_bin)
        if dc.empty:
            continue
        cents = compute_centroids(dc)
        if not cents: continue
        for cid, d_cluster in dc.groupby("cluster"):
            if cid==-1: continue
            centroid = cents[cid]
            cands = build_candidates_for_cluster(d_cluster, centroid, FUSION_RADARS, K_NEAREST_PER_RADAR)
            if len(cands)==0:
                continue

            # --- min-dmid selection ---
            dmid_values = []
            for c in cands:
                z = triplet_features_for_scoring(c, centroid)
                dmid_values.append(z[4])  # index 4 = dmid
            best_idx = int(np.argmin(dmid_values))
            best = cands[best_idx]
            # ----------------------------

            # rebuild per_radar table to pick actual rows by index
            per_radar={}
            for r in FUSION_RADARS:
                dr = d_cluster[d_cluster["radar"]==r]
                if dr.empty:
                    per_radar = {}
                    break
                per_radar[r]=nearest_k_to_centroid(dr, centroid, K_NEAREST_PER_RADAR).reset_index(drop=True)
            if len(per_radar) < 3:
                continue

            idx1, idx2, idx3 = best["rows"]
            sel = [per_radar[FUSION_RADARS[0]].iloc[idx1],
                   per_radar[FUSION_RADARS[1]].iloc[idx2],
                   per_radar[FUSION_RADARS[2]].iloc[idx3]]

            points = best["points"]; times = best["times"]
            trip = {"suffix": sfx, "tbin": int(tbin), "cluster": int(cid),
                    "x_avg": float(points[:,0].mean()), "y_avg": float(points[:,1].mean()),
                    "t_ref": float(np.mean(times))}
            for r, srow in zip(FUSION_RADARS, sel):
                for k in ["angle","range","inten","snr","noise"]:
                    trip[f"{r}_{k}"]=float(srow[k])
            rows.append(trip)
    return pd.DataFrame(rows)

# =============== Localization model (GBR) ===============

def build_Xy_from_triplets(df_sel: pd.DataFrame, feature_mode:str)->Tuple[np.ndarray,np.ndarray,List[str]]:
    if df_sel.empty: return np.empty((0,12)), np.empty((0,2)), []
    feat_cols=[]
    for r in FUSION_RADARS:
        if feature_mode=="angle":
            feat_cols += [f"{r}_angle", f"{r}_inten", f"{r}_snr", f"{r}_noise"]
        elif feature_mode=="range":
            feat_cols += [f"{r}_range", f"{r}_inten", f"{r}_snr", f"{r}_noise"]
        else:
            raise ValueError("FEATURE_MODE must be 'angle' or 'range'")
    X = df_sel[feat_cols].to_numpy(float)
    y = df_sel[["x_avg","y_avg"]].to_numpy(float)
    return X,y,feat_cols

def metrics_from_df(df: pd.DataFrame)->dict:
    dx = (df["x_pred"]-df["x_true"]).to_numpy(float)
    dy = (df["y_pred"]-df["y_true"]).to_numpy(float)
    err = np.hypot(dx,dy)
    return {
        "x_rmse": float(np.sqrt(np.mean(dx**2))),
        "y_rmse": float(np.sqrt(np.mean(dy**2))),
        "xy_rmse": float(np.sqrt(np.mean(err**2))),
        "xy_mae": float(np.mean(err)),
        "p90": float(np.percentile(err,90)),
        "N": int(len(df))
    }

def scatter_plot(y_true, y_pred, title, path):
    plt.figure(figsize=(5,5))
    plt.scatter(y_true[:,0], y_true[:,1], s=10, label="true")
    plt.scatter(y_pred[:,0], y_pred[:,1], s=10, label="pred")
    plt.axis("equal"); plt.title(title); plt.legend(); plt.tight_layout()
    savefig(path)

def hist_plot(y_true, y_pred, title, path):
    err=np.hypot(y_pred[:,0]-y_true[:,0], y_pred[:,1]-y_true[:,1])
    plt.figure(figsize=(6,4))
    plt.hist(err, bins=40, alpha=0.9)
    plt.xlabel("||error|| [m]"); plt.title(title); plt.tight_layout()
    savefig(path)

def eval_model_on_suffixes_gbr(gbr_predictor, X_builder, suffixes, feature_mode, tag, out_subdir):
    all_rows=[]
    for sfx in suffixes:
        sel = select_triplets_for_suffix(sfx)
        if sel.empty:
            print(f"[test {sfx}] no selected triplets")
            continue
        X_raw, y, feat_cols = X_builder(sel, feature_mode)
        Xs = gbr_predictor["prep"](X_raw)  # preprocessed X
        y_pred = gbr_predictor["pred"](Xs) # np array [N,2]

        df = pd.DataFrame({
            "suffix": sfx,
            "x_true": y[:,0], "y_true": y[:,1],
            "x_pred": y_pred[:,0], "y_pred": y_pred[:,1],
            "tbin": sel["tbin"].values, "cluster": sel["cluster"].values
        })
        all_rows.append(df)

        # save predictions CSV per suffix
        df.to_csv(os.path.join(out_subdir, f"{tag}_{feature_mode}_GBR_{sfx}_predictions.csv"), index=False)

        # plots: scatter + hist
        scatter_plot(y, y_pred,
                     f"Test {sfx} — GBR ({feature_mode})",
                     os.path.join(out_subdir, f"{tag}_{feature_mode}_GBR_{sfx}_test_scatter.png"))
        hist_plot(y, y_pred,
                  f"Test {sfx} — GBR ({feature_mode}) error",
                  os.path.join(out_subdir, f"{tag}_{feature_mode}_GBR_{sfx}_test_errhist.png"))

    return pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()

# =============== Helpers for CSM tuning ===============

def build_train_sel_for_current_csm(train_suffixes: List[str]) -> pd.DataFrame:
    """
    Build concatenated selected triplets for current global CSM configuration.
    """
    sel_rows=[]
    for sfx in train_suffixes:
        sel = select_triplets_for_suffix(sfx)
        if sel.empty:
            print(f"[CSM] {sfx}: 0 rows (current geometry)")
            continue
        sel_rows.append(sel)
        print(f"[CSM] {sfx}: {len(sel)} rows (current geometry)")
    return pd.concat(sel_rows, ignore_index=True) if sel_rows else pd.DataFrame()

def tune_csm_geometry(train_suffixes: List[str], tag_base: str):
    """
    Coarse grid search over CSM geometry parameters:
      - DBSCAN_EPS
      - DBSCAN_MINPTS
      - SPREAD_MAX_M
      - TIME_TOLERANCE_S

    Scoring:
      - Build triplets for all TRAIN_SUFFIXES.
      - Use feature_mode="angle".
      - Train a fixed GBR baseline.
      - Metric: XY-RMSE on validation.
    """
    # Must be declared BEFORE any assignment
    global DBSCAN_EPS, DBSCAN_MINPTS, SPREAD_MAX_M, TIME_TOLERANCE_S

    ensure_dir(OUT_DIR)
    out_subdir = os.path.join(OUT_DIR, f"{tag_base}_CSM_geometry_tuning")
    ensure_dir(out_subdir)

    geometry_grid = [
        {"eps": 0.15, "minpts": 3, "spread": 0.25, "time_tol": 0.30},
        {"eps": 0.20, "minpts": 3, "spread": 0.25, "time_tol": 0.30},     
        {"eps": 0.25, "minpts": 3, "spread": 0.25, "time_tol": 0.30},
        {"eps": 0.30, "minpts": 3, "spread": 0.30, "time_tol": 0.35},
        {"eps": 0.35, "minpts": 3, "spread": 0.35, "time_tol": 0.40},
        {"eps": 0.30, "minpts": 4, "spread": 0.30, "time_tol": 0.35},
        {"eps": 0.35, "minpts": 4, "spread": 0.35, "time_tol": 0.40},
        {"eps": 0.25, "minpts": 3, "spread": 0.4, "time_tol": 0.5},
        {"eps": 0.3, "minpts": 3, "spread": 0.4, "time_tol": 0.5},
        {"eps": 0.2, "minpts": 3, "spread": 0.4, "time_tol": 0.5},
        {"eps": 0.2, "minpts": 3, "spread": 0.45, "time_tol": 0.5},
        {"eps": 0.2, "minpts": 3, "spread": 0.45, "time_tol": 0.55},
        {"eps": 0.2, "minpts": 3, "spread": 0.5, "time_tol": 0.6},
        


    ]

    records=[]
    best_cfg=None
    best_rmse=float("inf")

    for i, cfg in enumerate(geometry_grid):
        print(f"\n[CSM tune] geometry config {i+1}/{len(geometry_grid)}: {cfg}")

        # Update global knobs
        DBSCAN_EPS       = cfg["eps"]
        DBSCAN_MINPTS    = cfg["minpts"]
        SPREAD_MAX_M     = cfg["spread"]
        TIME_TOLERANCE_S = cfg["time_tol"]

        train_sel = build_train_sel_for_current_csm(train_suffixes)
        n_rows = len(train_sel)

        if n_rows < 200:
            print(f"  [skip] only {n_rows} triplets; skipping config.")
            records.append({**cfg, "n_triplets": n_rows, "val_xy_rmse": np.nan})
            continue

        X_raw, y, _ = build_Xy_from_triplets(train_sel, "angle")
        Xtr_raw, Xval_raw, ytr, yval = train_test_split(
            X_raw, y, test_size=VAL_SPLIT, random_state=SEED
        )

        imp = SimpleImputer(strategy="mean")
        sca = StandardScaler()
        Xtr = sca.fit_transform(imp.fit_transform(Xtr_raw))
        Xval = sca.transform(imp.transform(Xval_raw))

        gbr_base = GradientBoostingRegressor(
            n_estimators=300,
            learning_rate=0.08,
            max_depth=3,
            subsample=0.9,
            loss="squared_error",
            random_state=SEED
        )
        gbr = MultiOutputRegressor(gbr_base)
        gbr.fit(Xtr, ytr)

        yval_pred = gbr.predict(Xval)
        df_va = pd.DataFrame({
            "x_true": yval[:,0], "y_true": yval[:,1],
            "x_pred": yval_pred[:,0], "y_pred": yval_pred[:,1]
        })
        m_va = metrics_from_df(df_va)
        val_rmse = m_va["xy_rmse"]

        print(f"  [CSM tune] val XY-RMSE = {val_rmse:.3f} m (n_triplets={n_rows})")

        records.append({**cfg, "n_triplets": n_rows, "val_xy_rmse": val_rmse})

        if np.isfinite(val_rmse) and val_rmse < best_rmse:
            best_rmse = val_rmse
            best_cfg = cfg.copy()

    df_geom = pd.DataFrame(records)
    df_geom.to_csv(os.path.join(out_subdir, f"{tag_base}_CSM_geometry_grid.csv"), index=False)

    if best_cfg is None:
        print("[CSM tune] No valid configuration found. Using defaults.")
        return None

    # Set best globals
    DBSCAN_EPS       = best_cfg["eps"]
    DBSCAN_MINPTS    = best_cfg["minpts"]
    SPREAD_MAX_M     = best_cfg["spread"]
    TIME_TOLERANCE_S = best_cfg["time_tol"]

    with open(os.path.join(out_subdir, f"{tag_base}_CSM_geometry_best.txt"), "w") as f:
        f.write("Best CSM geometry configuration:\n")
        for k, v in best_cfg.items():
            f.write(f"{k}: {v}\n")
        f.write(f"\nVal XY-RMSE: {best_rmse:.6f} m\n")

    print(f"[CSM tune] Best config = {best_cfg}  (XY-RMSE={best_rmse:.3f} m)")
    return best_cfg

# =============== GBR hyperparameter optimisation ===============

def tune_gbr_hyperparams(Xtr_raw: np.ndarray,
                         ytr: np.ndarray,
                         Xval_raw: np.ndarray,
                         yval: np.ndarray,
                         tag_base: str,
                         feature_mode: str,
                         out_subdir: str) -> dict:
    """
    Simple manual grid search for GBR hyperparameters using train/val split.
    Selection metric: XY-RMSE on validation.
    Returns a dict with:
        "params"         - best hyperparameter dict
        "imp"            - fitted imputer
        "sca"            - fitted scaler
        "model"          - fitted MultiOutputRegressor(GBR)
        "train_metrics"  - metrics on train
        "val_metrics"    - metrics on val
    Also saves a CSV with all tried configs + their metrics.
    """
    # Grid can be extended if you want more exhaustive search
    param_grid = [
        {"n_estimators": 200, "learning_rate": 0.05, "max_depth": 2, "subsample": 0.8},
        {"n_estimators": 300, "learning_rate": 0.05, "max_depth": 3, "subsample": 0.8},
        {"n_estimators": 400, "learning_rate": 0.10, "max_depth": 3, "subsample": 0.9},
        {"n_estimators": 600, "learning_rate": 0.05, "max_depth": 4, "subsample": 0.9},
        {"n_estimators": 600, "learning_rate": 0.1, "max_depth": 3, "subsample": 0.9},
        {"n_estimators": 600, "learning_rate": 0.15, "max_depth": 4, "subsample": 0.9},
        {"n_estimators": 800, "learning_rate": 0.05, "max_depth": 1, "subsample": 0.9},
        {"n_estimators": 800, "learning_rate": 0.1, "max_depth": 3, "subsample": 0.9},
        {"n_estimators": 1000, "learning_rate": 0.1, "max_depth": 3, "subsample": 0.9},
        {"n_estimators": 1000, "learning_rate": 0.05, "max_depth": 4, "subsample": 0.9},
        {"n_estimators": 1500, "learning_rate": 0.05, "max_depth": 4, "subsample": 0.9},
        {"n_estimators": 1500, "learning_rate": 0.1, "max_depth": 4, "subsample": 0.9},
        {"n_estimators": 2000, "learning_rate": 0.05, "max_depth": 3, "subsample": 0.9},
        {"n_estimators": 2500, "learning_rate": 0.05, "max_depth": 4, "subsample": 0.9},
        {"n_estimators": 3000, "learning_rate": 0.05, "max_depth": 4, "subsample": 0.9},


    ]

    results = []
    best_cfg = None
    best_val_rmse = float("inf")

    for i, params in enumerate(param_grid):
        print(f"[GBR tune {feature_mode}] config {i+1}/{len(param_grid)}: {params}")

        imp = SimpleImputer(strategy="mean")
        sca = StandardScaler()

        Xtr0 = imp.fit_transform(Xtr_raw)
        Xval0 = imp.transform(Xval_raw)
        Xtr = sca.fit_transform(Xtr0)
        Xval = sca.transform(Xval0)

        gbr_base = GradientBoostingRegressor(
            n_estimators=params["n_estimators"],
            learning_rate=params["learning_rate"],
            max_depth=params["max_depth"],
            subsample=params["subsample"],
            loss="squared_error",
            random_state=SEED
        )
        gbr = MultiOutputRegressor(gbr_base)
        gbr.fit(Xtr, ytr)

        # Train metrics
        ytr_pred = gbr.predict(Xtr)
        df_tr = pd.DataFrame({
            "x_true": ytr[:,0], "y_true": ytr[:,1],
            "x_pred": ytr_pred[:,0], "y_pred": ytr_pred[:,1]
        })
        m_tr = metrics_from_df(df_tr)

        # Val metrics
        yval_pred = gbr.predict(Xval)
        df_va = pd.DataFrame({
            "x_true": yval[:,0], "y_true": yval[:,1],
            "x_pred": yval_pred[:,0], "y_pred": yval_pred[:,1]
        })
        m_va = metrics_from_df(df_va)

        row = {
            "n_estimators": params["n_estimators"],
            "learning_rate": params["learning_rate"],
            "max_depth": params["max_depth"],
            "subsample": params["subsample"],
            "train_xy_rmse": m_tr["xy_rmse"],
            "train_xy_mae":  m_tr["xy_mae"],
            "val_xy_rmse":   m_va["xy_rmse"],
            "val_xy_mae":    m_va["xy_mae"],
        }
        results.append(row)

        if m_va["xy_rmse"] < best_val_rmse:
            best_val_rmse = m_va["xy_rmse"]
            best_cfg = {
                "params": params,
                "imp": imp,
                "sca": sca,
                "model": gbr,
                "train_metrics": m_tr,
                "val_metrics": m_va,
            }

    # Save grid search results
    df_grid = pd.DataFrame(results)
    df_grid.to_csv(os.path.join(
        out_subdir,
        f"{tag_base}_{feature_mode}_GBR_hyperparam_grid.csv"
    ), index=False)

    print(f"[GBR tune {feature_mode}] best val XY-RMSE={best_val_rmse:.3f} with params={best_cfg['params']}")

    # Save best params as a small text file
    best_txt_path = os.path.join(
        out_subdir,
        f"{tag_base}_{feature_mode}_GBR_best_params.txt"
    )
    with open(best_txt_path, "w") as f:
        f.write(f"Best params (feature_mode={feature_mode}):\n")
        for k,v in best_cfg["params"].items():
            f.write(f"{k}: {v}\n")
        f.write(f"\nBest val XY-RMSE: {best_val_rmse:.6f}\n")

    return best_cfg

# =============== Runner for one feature_mode ===============

def run_gbr_for_feature_mode(train_sel: pd.DataFrame, feature_mode: str, tag_base: str):
    """
    Train/eval GBR (squared error) for one feature_mode.
    Steps:
      - Build X,y from selected triplets.
      - Train/val split.
      - Hyperparameter tuning (manual grid search) using val XY-RMSE.
      - Save:
          * hyperparameter grid results
          * best-GBR train/val metrics CSV + bar plot
          * best hyperparameters (TXT)
          * test metrics CSV (all + cc2-only)
          * predictions & plots for TEST_SUFFIXES
    """
    ensure_dir(OUT_DIR)
    out_subdir = os.path.join(OUT_DIR, f"{tag_base}_{feature_mode}_GBR")
    ensure_dir(out_subdir)

    # === Build X,y
    X_raw, y, feat_cols = build_Xy_from_triplets(train_sel, feature_mode)
    if len(X_raw)==0:
        raise RuntimeError(f"No training rows for feature_mode={feature_mode}")

    # shared train/val split
    Xtr_raw, Xval_raw, ytr, yval = train_test_split(X_raw, y, test_size=VAL_SPLIT, random_state=SEED)

    # === Hyperparameter optimisation for GBR
    best_cfg = tune_gbr_hyperparams(
        Xtr_raw, ytr, Xval_raw, yval,
        tag_base=tag_base,
        feature_mode=feature_mode,
        out_subdir=out_subdir
    )

    imp_gb = best_cfg["imp"]
    sca_gb = best_cfg["sca"]
    gbr     = best_cfg["model"]
    m_tr    = best_cfg["train_metrics"]
    m_va    = best_cfg["val_metrics"]

    def gbr_prep(X):
        X0 = imp_gb.transform(X)
        return sca_gb.transform(X0)

    gbr_predictor = {
        "name": "GBR",
        "prep": gbr_prep,
        "pred": lambda Xs: gbr.predict(Xs)
    }

    # === Train/val metrics (from best config)
    ytr_pred = gbr_predictor["pred"](gbr_predictor["prep"](Xtr_raw))
    yval_pred = gbr_predictor["pred"](gbr_predictor["prep"](Xval_raw))

    df_tr = pd.DataFrame({
        "x_true": ytr[:,0], "y_true": ytr[:,1],
        "x_pred": ytr_pred[:,0], "y_pred": ytr_pred[:,1]
    })
    df_va = pd.DataFrame({
        "x_true": yval[:,0], "y_true": yval[:,1],
        "x_pred": yval_pred[:,0], "y_pred": yval_pred[:,1]
    })

    # re-compute metrics for consistency (or use m_tr/m_va directly)
    m_tr = metrics_from_df(df_tr)
    m_va = metrics_from_df(df_va)

    df_tv = pd.DataFrame([
        {"subset": "train", **m_tr},
        {"subset": "val",   **m_va}
    ])
    df_tv.to_csv(os.path.join(out_subdir,
        f"{tag_base}_{feature_mode}_GBR_train_val_metrics.csv"), index=False)

    # bar plot: train vs val XY RMSE & MAE
    rmse_vals = [m_tr["xy_rmse"], m_va["xy_rmse"]]
    mae_vals  = [m_tr["xy_mae"],  m_va["xy_mae"]]
    labels    = ["train", "val"]

    x = np.arange(len(labels))
    width = 0.35

    plt.figure(figsize=(6,4))
    plt.bar(x - width/2, rmse_vals, width, label="XY RMSE")
    plt.bar(x + width/2, mae_vals,  width, label="XY MAE")
    plt.xticks(x, labels)
    plt.ylabel("Error [m]")
    plt.title(f"{tag_base} ({feature_mode}) — GBR train/val metrics (tuned)")
    plt.legend()
    plt.tight_layout()
    savefig(os.path.join(out_subdir,
        f"{tag_base}_{feature_mode}_GBR_train_val_bar.png"))

    # === Evaluate on TEST suffixes with the tuned model
    df_test = eval_model_on_suffixes_gbr(gbr_predictor, build_Xy_from_triplets,
                                         TEST_SUFFIXES, feature_mode,
                                         tag=tag_base, out_subdir=out_subdir)

    metrics_rows=[]
    if not df_test.empty:
        m_all = metrics_from_df(df_test)
        metrics_rows.append({"feature_mode":feature_mode, "model":"GBR", **m_all})
        df_cc2 = df_test[df_test["suffix"]=="cc2"]
        if not df_cc2.empty:
            m_cc2 = metrics_from_df(df_cc2)
            metrics_rows.append({"feature_mode":feature_mode, "model":"GBR_cc2_only", **m_cc2})

    if metrics_rows:
        df_metrics = pd.DataFrame(metrics_rows)
        df_metrics.to_csv(os.path.join(out_subdir,
            f"{tag_base}_{feature_mode}_GBR_test_metrics.csv"), index=False)

# =============== Main ===============

def main():
    set_seed(SEED)
    ensure_dir(OUT_DIR)
    print(f"[config] device={DEVICE}")
    print("[info] Triplet selector = min-dmid (geometric heuristic)")
    print("[info] Localization model = GBR (squared_error) with hyperparameter tuning")

    # 0) Tune CSM geometry (DBSCAN + gates) using angle-mode + GBR baseline
    print("\n[step] Tuning CSM geometry (DBSCAN_EPS, MINPTS, SPREAD_MAX_M, TIME_TOLERANCE_S)...")
    best_geom = tune_csm_geometry(TRAIN_SUFFIXES, tag_base="CSM")
    if best_geom is not None:
        print(f"[info] Using tuned CSM geometry: {best_geom}")
    else:
        print("[info] Using default CSM geometry (tuning failed or skipped).")

    # 1) Build selected triplets for TRAIN using final CSM geometry
    print("\n[step] Building train-selected triplets with final CSM geometry...")
    sel_rows=[]
    for sfx in TRAIN_SUFFIXES:
        sel = select_triplets_for_suffix(sfx)
        if sel.empty:
            print(f"[train sel] {sfx}: 0 rows"); continue
        sel_rows.append(sel)
        print(f"[train sel] {sfx}: {len(sel)} rows")
    if not sel_rows:
        print("[abort] no selected triplets on TRAIN."); return
    train_sel = pd.concat(sel_rows, ignore_index=True)
    train_sel.to_csv(os.path.join(OUT_DIR, "train_selected_triplets_min_dmid.csv"), index=False)

    # 2) Run GBR for BOTH feature modes (angle & range)
    for FEATURE_MODE in ["angle","range"]:
        print(f"\n=== Running GBR for FEATURE_MODE={FEATURE_MODE} ===")
        run_gbr_for_feature_mode(train_sel, FEATURE_MODE, tag_base="CSM")

if __name__=="__main__":
    main()


[config] device=cuda
[info] Triplet selector = min-dmid (geometric heuristic)
[info] Localization model = GBR (squared_error) with hyperparameter tuning

[step] Tuning CSM geometry (DBSCAN_EPS, MINPTS, SPREAD_MAX_M, TIME_TOLERANCE_S)...

[CSM tune] geometry config 1/13: {'eps': 0.15, 'minpts': 3, 'spread': 0.25, 'time_tol': 0.3}
[CSM] rr1: 44 rows (current geometry)
[CSM] rr2: 30 rows (current geometry)
[CSM] rr3: 49 rows (current geometry)
[CSM] rr4: 35 rows (current geometry)
[CSM] rr5: 38 rows (current geometry)
[CSM] rr6: 35 rows (current geometry)
[CSM] rr7: 29 rows (current geometry)
[CSM] rr8: 49 rows (current geometry)
[CSM] rr9: 54 rows (current geometry)
[CSM] rr12: 40 rows (current geometry)
[CSM] rr11: 40 rows (current geometry)
[CSM] cc3: 35 rows (current geometry)
[CSM] cc5: 47 rows (current geometry)
[CSM] cc6: 38 rows (current geometry)
[CSM] cc7: 28 rows (current geometry)
[CSM] cc8: 39 rows (current geometry)
[CSM] pcc3: 85 rows (current geometry)
[CSM] prr1: 38 rows 

In [1]:
# -*- coding: utf-8 -*-
"""
CSM triplet selection (min-dmid heuristic) + GBR regressor
---------------------------------------------------------
- Triplet selection: deterministic geometric "scorer":
    * Among all candidates per cluster, choose the one whose midpoint
      is closest to the DBSCAN cluster centroid (min dmid).
- Feature switch: angle/range (plus inten, snr, noise per radar).
- Localization model: GradientBoostingRegressor (squared_error), wrapped
  with MultiOutputRegressor for (x, y).
- Hyperparameter optimisation:
    * CSM geometry (DBSCAN_EPS, DBSCAN_MINPTS, SPREAD_MAX_M, TIME_TOLERANCE_S)
      via angle-mode + fixed GBR baseline.
    * GBR hyperparameters via manual grid search per feature_mode (angle/range).
- Saves:
    * Selected training triplets (CSV) for final tuned CSM.
    * CSM geometry grid + best config (CSV/TXT).
    * Train/val metrics for best GBR (CSV & bar plots).
    * GBR hyperparameter grid per feature_mode (CSV).
    * Test metrics (CSV; incl. cc2-only).
    * Test scatter + error histogram for each feature_mode.
"""

import os, glob, math, random
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import DBSCAN
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor

import torch

plt.rcParams.update({
    "figure.dpi": 120, "axes.grid": True, "grid.linestyle": "--",
    "pdf.fonttype": 42, "ps.fonttype": 42
})

# ======================
# === Configuration  ===
# ======================

BASE_PATH = "/home/charithag/master thesis/practise/18Oct/corrected"

# Sessions
TRAIN_SUFFIXES = [
    "rr1","rr2","rr3","rr4","rr5","rr6","rr7","rr8","rr9","rr12","rr11",
    "cc3","cc5","cc6","cc7","cc8","pcc3","prr1","prr2","prr3","prr4",
    "pcc1","pcc2","cc11","cc12","cc13","cc10","cc1","cc2"
]
TEST_SUFFIXES  = ["cc2"]  # explicit cc2 test plots

# Radar roles
FUSION_RADARS   = ["rpi4","rpi1","rpi3"]
REFERENCE_RADARS= ["radar1","rpi2"]  # kept for completeness (not used here)

# Anchors + yaws (deg CCW+)
radar_positions: Dict[str, Tuple[float, float]] = {
    "rpi4": (0.0, 0.0),
    "rpi1": (0.07, 0.7),
    "rpi3": (-0.04, -0.75),
    "rpi5": (2.3, 0.4),
    "rpi6": (2.8, 1.7),
}
sides = {r: {"angle": 0.0} for r in radar_positions.keys()}

# File column mapping (ti_mmwave-style, 0-based)
CSV_X_COL        = 3
CSV_Y_COL        = 4
CSV_RANGE_COL    = 6
CSV_DOPPLER_COL  = 8
CSV_ANGLE_COL    = 9          # (1-based col10)
CSV_INTEN_COL    = 10         # (1-based col11)
CSV_SNR_COL      = 11         # (1-based col12)
CSV_NOISE_COL    = 12         # (1-based col13)
CSV_TIME_COL     = -1

# Pipeline knobs (initial defaults; may be tuned)
BIN_SECONDS      = 0.20
DBSCAN_EPS       = 0.30
DBSCAN_MINPTS    = 3
TIME_TOLERANCE_S = 0.35
SPREAD_MAX_M     = 0.30
K_NEAREST_PER_RADAR = 3
REMOVE_DOPPLER_EQ: Optional[float] = 8.0

# Training knobs (GBR)
SEED        = 1337
VAL_SPLIT   = 0.15

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Output
OUT_DIR = "ML results"  # (user requested this exact folder name)

# ======================
# === Utilities      ===
# ======================

def ensure_dir(p): os.makedirs(p, exist_ok=True)
def savefig(path): ensure_dir(os.path.dirname(path)); plt.savefig(path, dpi=300, bbox_inches="tight"); plt.close()

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def _has_header(path: str) -> bool:
    try:
        with open(path,"r",errors="ignore") as f:
            toks = (f.readline().strip()).split(",")
        for t in toks:
            try: float(t)
            except ValueError: return True
        return False
    except: return False

def _read_any(path: str, header: Optional[int]) -> pd.DataFrame:
    try:    return pd.read_csv(path, header=header, sep=None, engine="python")
    except: return pd.read_csv(path, header=header, delim_whitespace=True)

def _get(df, spec):
    if isinstance(spec,int):
        idx = spec if spec>=0 else df.shape[1]+spec
        return df.iloc[:,idx]
    if isinstance(spec,str):
        if spec in df.columns: return df[spec]
        low = {c.lower():c for c in df.columns}
        if spec.lower() in low: return df[low[spec.lower()]]
    raise KeyError(spec)

def discover_files_for_radar(base: str, radar: str, suffix: str)->List[str]:
    exact = os.path.join(base, radar, suffix)
    if os.path.isfile(exact): return [exact]
    if os.path.isdir(exact):
        files = glob.glob(os.path.join(exact,"**","*"), recursive=True)
        return sorted([f for f in files if os.path.isfile(f)])
    root = os.path.join(base, radar)
    if os.path.isdir(root):
        candidates = glob.glob(os.path.join(root,"**","*"), recursive=True)
        hits = [f for f in candidates if os.path.isfile(f) and os.path.basename(f)==suffix]
        return sorted(hits)
    return []

def load_one_file(path: str) -> pd.DataFrame:
    header = 0 if _has_header(path) else None
    df = _read_any(path, header)
    out = pd.DataFrame({
        "x":        pd.to_numeric(_get(df, CSV_X_COL), errors="coerce"),
        "y":        pd.to_numeric(_get(df, CSV_Y_COL), errors="coerce"),
        "range":    pd.to_numeric(_get(df, CSV_RANGE_COL), errors="coerce"),
        "angle":    pd.to_numeric(_get(df, CSV_ANGLE_COL), errors="coerce"),
        "inten":    pd.to_numeric(_get(df, CSV_INTEN_COL), errors="coerce"),
        "snr":      pd.to_numeric(_get(df, CSV_SNR_COL), errors="coerce"),
        "noise":    pd.to_numeric(_get(df, CSV_NOISE_COL), errors="coerce"),
    })
    try: out["doppler"] = pd.to_numeric(_get(df, CSV_DOPPLER_COL), errors="coerce")
    except: out["doppler"] = np.nan
    try: out["timestamp"] = _get(df, CSV_TIME_COL)
    except: out["timestamp"] = np.nan
    return out.dropna(subset=["x","y"]).reset_index(drop=True)

def load_radar_suffix(base: str, radar: str, suffix: str) -> pd.DataFrame:
    files = discover_files_for_radar(base, radar, suffix)
    if not files:
        return pd.DataFrame(columns=["x","y","range","angle","inten","snr","noise","timestamp","radar"])
    parts=[]
    for f in files:
        try:
            df = load_one_file(f)
            if REMOVE_DOPPLER_EQ is not None and "doppler" in df.columns:
                df = df[~np.isclose(df["doppler"].astype(float), REMOVE_DOPPLER_EQ)]
            df["radar"]=radar
            parts.append(df)
        except Exception as e:
            print(f"[warn] load {f}: {e}")
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

def rot2d(theta):
    c,s=math.cos(theta), math.sin(theta)
    return np.array([[c,-s],[s,c]],float)

def transform_local_to_world(df: pd.DataFrame, yaw_deg: float, tx: float, ty: float)->pd.DataFrame:
    R = rot2d(math.radians(yaw_deg))
    xy = df[["x","y"]].to_numpy(float) @ R.T
    out = df.copy()
    out["xw"]=xy[:,0]+tx; out["yw"]=xy[:,1]+ty
    return out

def load_transform_suffix_all_radars(sfx: str) -> pd.DataFrame:
    parts=[]
    for r,(tx,ty) in radar_positions.items():
        d = load_radar_suffix(BASE_PATH, r, sfx)
        if d.empty: continue
        yaw = float(sides.get(r,{}).get("angle",0.0))
        d = transform_local_to_world(d, yaw, tx, ty)
        parts.append(d)
    if not parts: return pd.DataFrame()
    df = pd.concat(parts, ignore_index=True)
    return df[["x","y","range","angle","inten","snr","noise","timestamp","radar","xw","yw"]]

def to_seconds(s: pd.Series)->pd.Series:
    a = s.astype(str).str.extract(r'([0-9]+(?:\.[0-9]+)?)')[0]
    return pd.to_numeric(a, errors="coerce")

# =============== Binning + Clustering ===============

def bin_time(df: pd.DataFrame, bin_s: float)->pd.DataFrame:
    df = df.dropna(subset=["timestamp"]).copy()
    df["t"] = to_seconds(df["timestamp"])
    df = df.dropna(subset=["t"])
    df["tbin"] = np.floor(df["t"]/bin_s).astype(int)
    return df

def cluster_bin(df_bin: pd.DataFrame)->pd.DataFrame:
    if df_bin.empty: return pd.DataFrame()
    X = df_bin[["xw","yw"]].to_numpy(float)
    labels = DBSCAN(eps=DBSCAN_EPS, min_samples=DBSCAN_MINPTS).fit_predict(X)
    out = df_bin.copy()
    out["cluster"]=labels
    return out

def compute_centroids(df_bin: pd.DataFrame)->Dict[int, Tuple[float,float]]:
    cents={}
    for c, d in df_bin.groupby("cluster"):
        if c==-1: continue
        cents[c]=(float(d["xw"].mean()), float(d["yw"].mean()))
    return cents

# =============== Candidate triplets per cluster ===============

def nearest_k_to_centroid(d: pd.DataFrame, centroid: Tuple[float,float], k:int)->pd.DataFrame:
    dx = d["xw"].to_numpy()-centroid[0]
    dy = d["yw"].to_numpy()-centroid[1]
    dists = np.hypot(dx,dy)
    idx = np.argsort(dists)[:min(k,len(d))]
    return d.iloc[idx].copy()

def build_candidates_for_cluster(d_cluster: pd.DataFrame,
                                 centroid: Tuple[float,float],
                                 fusion_radars: List[str],
                                 k_each:int=K_NEAREST_PER_RADAR)->List[dict]:
    per_radar={}
    for r in fusion_radars:
        dr = d_cluster[d_cluster["radar"]==r]
        if dr.empty:
            return []  # must have all fusion radars
        kn = nearest_k_to_centroid(dr, centroid, k_each).reset_index(drop=True)
        per_radar[r]=kn

    R1,R2,R3 = fusion_radars[:3]
    cands=[]
    n1, n2, n3 = len(per_radar[R1]), len(per_radar[R2]), len(per_radar[R3])
    for ia in range(n1):
        a = per_radar[R1].iloc[ia]
        for ib in range(n2):
            b = per_radar[R2].iloc[ib]
            for ic in range(n3):
                c = per_radar[R3].iloc[ic]
                # time gate
                t_ok = (abs(a["t"]-b["t"])<=TIME_TOLERANCE_S and
                        abs(a["t"]-c["t"])<=TIME_TOLERANCE_S)
                if not t_ok:
                    continue
                # spread gate
                P = np.array([[a["xw"],a["yw"]],
                              [b["xw"],b["yw"]],
                              [c["xw"],c["yw"]]], float)
                d01 = np.hypot(*(P[0]-P[1])); d02 = np.hypot(*(P[0]-P[2])); d12 = np.hypot(*(P[1]-P[2]))
                spread = max(d01,d02,d12)
                if spread>SPREAD_MAX_M:
                    continue
                cands.append({
                    "rows": (ia, ib, ic),
                    "points": P,                 # [3,2] world coords
                    "times": (float(a["t"]), float(b["t"]), float(c["t"])),
                    "centroid": (float(P[:,0].mean()), float(P[:,1].mean()))
                })
    return cands

# =============== Triplet features (for min-dmid) ===============

def triplet_features_for_scoring(cand: dict, cluster_centroid: Tuple[float,float])->np.ndarray:
    """
    z = [d01, d02, d12, spread, dmid, |t0-t1|, |t0-t2|, |t1-t2|]
    We use index 4 (dmid) to implement the min-dmid selector.
    """
    P = cand["points"]; t0,t1,t2 = cand["times"]
    d01 = np.hypot(*(P[0]-P[1])); d02 = np.hypot(*(P[0]-P[2])); d12 = np.hypot(*(P[1]-P[2]))
    spread = max(d01,d02,d12)
    mid = P.mean(axis=0)
    dmid = np.hypot(*(mid - np.array(cluster_centroid)))
    dt = np.array([abs(t0-t1), abs(t0-t2), abs(t1-t2)], float)
    z = np.array([d01,d02,d12,spread,dmid, dt[0],dt[1],dt[2]], float)
    return z

# =============== Selection using min-dmid heuristic ===============

def select_triplets_for_suffix(sfx: str)->pd.DataFrame:
    """
    Deterministic triplet selection for a suffix:
    - Time-bin + DBSCAN per bin.
    - For each cluster:
        * Build candidate triplets (K-nearest per radar, time/spread gates).
        * For each candidate, compute features z.
        * Select candidate with minimal dmid (midpoint-to-cluster-centroid distance).
    Uses global CSM geometry knobs: BIN_SECONDS, DBSCAN_EPS, DBSCAN_MINPTS,
    SPREAD_MAX_M, TIME_TOLERANCE_S, K_NEAREST_PER_RADAR.
    """
    df = load_transform_suffix_all_radars(sfx)
    if df.empty: return pd.DataFrame()
    df = bin_time(df, BIN_SECONDS)
    rows=[]
    for tbin, d_bin in df.groupby("tbin"):
        dc = cluster_bin(d_bin)
        if dc.empty:
            continue
        cents = compute_centroids(dc)
        if not cents: continue
        for cid, d_cluster in dc.groupby("cluster"):
            if cid==-1: continue
            centroid = cents[cid]
            cands = build_candidates_for_cluster(d_cluster, centroid, FUSION_RADARS, K_NEAREST_PER_RADAR)
            if len(cands)==0:
                continue

            # --- min-dmid selection ---
            dmid_values = []
            for c in cands:
                z = triplet_features_for_scoring(c, centroid)
                dmid_values.append(z[4])  # index 4 = dmid
            best_idx = int(np.argmin(dmid_values))
            best = cands[best_idx]
            # ----------------------------

            # rebuild per_radar table to pick actual rows by index
            per_radar={}
            for r in FUSION_RADARS:
                dr = d_cluster[d_cluster["radar"]==r]
                if dr.empty:
                    per_radar = {}
                    break
                per_radar[r]=nearest_k_to_centroid(dr, centroid, K_NEAREST_PER_RADAR).reset_index(drop=True)
            if len(per_radar) < 3:
                continue

            idx1, idx2, idx3 = best["rows"]
            sel = [per_radar[FUSION_RADARS[0]].iloc[idx1],
                   per_radar[FUSION_RADARS[1]].iloc[idx2],
                   per_radar[FUSION_RADARS[2]].iloc[idx3]]

            points = best["points"]; times = best["times"]
            trip = {"suffix": sfx, "tbin": int(tbin), "cluster": int(cid),
                    "x_avg": float(points[:,0].mean()), "y_avg": float(points[:,1].mean()),
                    "t_ref": float(np.mean(times))}
            for r, srow in zip(FUSION_RADARS, sel):
                for k in ["angle","range","inten","snr","noise"]:
                    trip[f"{r}_{k}"]=float(srow[k])
            rows.append(trip)
    return pd.DataFrame(rows)

# =============== Localization model (GBR) ===============

def build_Xy_from_triplets(df_sel: pd.DataFrame, feature_mode:str)->Tuple[np.ndarray,np.ndarray,List[str]]:
    if df_sel.empty: return np.empty((0,12)), np.empty((0,2)), []
    feat_cols=[]
    for r in FUSION_RADARS:
        if feature_mode=="angle":
            feat_cols += [f"{r}_angle", f"{r}_inten", f"{r}_snr", f"{r}_noise"]
        elif feature_mode=="range":
            feat_cols += [f"{r}_range", f"{r}_inten", f"{r}_snr", f"{r}_noise"]
        else:
            raise ValueError("FEATURE_MODE must be 'angle' or 'range'")
    X = df_sel[feat_cols].to_numpy(float)
    y = df_sel[["x_avg","y_avg"]].to_numpy(float)
    return X,y,feat_cols

def metrics_from_df(df: pd.DataFrame)->dict:
    dx = (df["x_pred"]-df["x_true"]).to_numpy(float)
    dy = (df["y_pred"]-df["y_true"]).to_numpy(float)
    err = np.hypot(dx,dy)
    return {
        "x_rmse": float(np.sqrt(np.mean(dx**2))),
        "y_rmse": float(np.sqrt(np.mean(dy**2))),
        "xy_rmse": float(np.sqrt(np.mean(err**2))),
        "xy_mae": float(np.mean(err)),
        "p90": float(np.percentile(err,90)),
        "N": int(len(df))
    }

def scatter_plot(y_true, y_pred, title, path):
    plt.figure(figsize=(5,5))
    plt.scatter(y_true[:,0], y_true[:,1], s=10, label="true")
    plt.scatter(y_pred[:,0], y_pred[:,1], s=10, label="pred")
    plt.axis("equal"); plt.title(title); plt.legend(); plt.tight_layout()
    savefig(path)

def hist_plot(y_true, y_pred, title, path):
    err=np.hypot(y_pred[:,0]-y_true[:,0], y_pred[:,1]-y_true[:,1])
    plt.figure(figsize=(6,4))
    plt.hist(err, bins=40, alpha=0.9)
    plt.xlabel("||error|| [m]"); plt.title(title); plt.tight_layout()
    savefig(path)

def eval_model_on_suffixes_gbr(gbr_predictor, X_builder, suffixes, feature_mode, tag, out_subdir):
    all_rows=[]
    for sfx in suffixes:
        sel = select_triplets_for_suffix(sfx)
        if sel.empty:
            print(f"[test {sfx}] no selected triplets")
            continue
        X_raw, y, feat_cols = X_builder(sel, feature_mode)
        Xs = gbr_predictor["prep"](X_raw)  # preprocessed X
        y_pred = gbr_predictor["pred"](Xs) # np array [N,2]

        df = pd.DataFrame({
            "suffix": sfx,
            "x_true": y[:,0], "y_true": y[:,1],
            "x_pred": y_pred[:,0], "y_pred": y_pred[:,1],
            "tbin": sel["tbin"].values, "cluster": sel["cluster"].values
        })
        all_rows.append(df)

        # save predictions CSV per suffix
        df.to_csv(os.path.join(out_subdir, f"{tag}_{feature_mode}_GBR_{sfx}_predictions.csv"), index=False)

        # plots: scatter + hist
        scatter_plot(y, y_pred,
                     f"Test {sfx} — GBR ({feature_mode})",
                     os.path.join(out_subdir, f"{tag}_{feature_mode}_GBR_{sfx}_test_scatter.png"))
        hist_plot(y, y_pred,
                  f"Test {sfx} — GBR ({feature_mode}) error",
                  os.path.join(out_subdir, f"{tag}_{feature_mode}_GBR_{sfx}_test_errhist.png"))

    return pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()

# =============== Helpers for CSM tuning ===============

def build_train_sel_for_current_csm(train_suffixes: List[str]) -> pd.DataFrame:
    """
    Build concatenated selected triplets for current global CSM configuration.
    """
    sel_rows=[]
    for sfx in train_suffixes:
        sel = select_triplets_for_suffix(sfx)
        if sel.empty:
            print(f"[CSM] {sfx}: 0 rows (current geometry)")
            continue
        sel_rows.append(sel)
        print(f"[CSM] {sfx}: {len(sel)} rows (current geometry)")
    return pd.concat(sel_rows, ignore_index=True) if sel_rows else pd.DataFrame()

def tune_csm_geometry(train_suffixes: List[str], tag_base: str):
    """
    Coarse grid search over CSM geometry parameters:
      - DBSCAN_EPS
      - DBSCAN_MINPTS
      - SPREAD_MAX_M
      - TIME_TOLERANCE_S

    Scoring:
      - Build triplets for all TRAIN_SUFFIXES.
      - Use feature_mode="angle".
      - Train a fixed GBR baseline.
      - Metric: XY-RMSE on validation.
    """
    # Must be declared BEFORE any assignment
    global DBSCAN_EPS, DBSCAN_MINPTS, SPREAD_MAX_M, TIME_TOLERANCE_S

    ensure_dir(OUT_DIR)
    out_subdir = os.path.join(OUT_DIR, f"{tag_base}_CSM_geometry_tuning")
    ensure_dir(out_subdir)

    geometry_grid = [
        {"eps": 0.15, "minpts": 3, "spread": 0.25, "time_tol": 0.30},
        {"eps": 0.20, "minpts": 3, "spread": 0.25, "time_tol": 0.30},     
        {"eps": 0.25, "minpts": 3, "spread": 0.25, "time_tol": 0.30},
        {"eps": 0.30, "minpts": 3, "spread": 0.30, "time_tol": 0.35},
        {"eps": 0.35, "minpts": 3, "spread": 0.35, "time_tol": 0.40},
        {"eps": 0.30, "minpts": 4, "spread": 0.30, "time_tol": 0.35},
        {"eps": 0.35, "minpts": 4, "spread": 0.35, "time_tol": 0.40},
    ]

    records=[]
    best_cfg=None
    best_rmse=float("inf")

    for i, cfg in enumerate(geometry_grid):
        print(f"\n[CSM tune] geometry config {i+1}/{len(geometry_grid)}: {cfg}")

        # Update global knobs
        DBSCAN_EPS       = cfg["eps"]
        DBSCAN_MINPTS    = cfg["minpts"]
        SPREAD_MAX_M     = cfg["spread"]
        TIME_TOLERANCE_S = cfg["time_tol"]

        train_sel = build_train_sel_for_current_csm(train_suffixes)
        n_rows = len(train_sel)

        if n_rows < 200:
            print(f"  [skip] only {n_rows} triplets; skipping config.")
            records.append({**cfg, "n_triplets": n_rows, "val_xy_rmse": np.nan})
            continue

        X_raw, y, _ = build_Xy_from_triplets(train_sel, "angle")
        Xtr_raw, Xval_raw, ytr, yval = train_test_split(
            X_raw, y, test_size=VAL_SPLIT, random_state=SEED
        )

        imp = SimpleImputer(strategy="mean")
        sca = StandardScaler()
        Xtr = sca.fit_transform(imp.fit_transform(Xtr_raw))
        Xval = sca.transform(imp.transform(Xval_raw))

        gbr_base = GradientBoostingRegressor(
            n_estimators=300,
            learning_rate=0.08,
            max_depth=3,
            subsample=0.9,
            loss="absolute_error",
            random_state=SEED
        )
        gbr = MultiOutputRegressor(gbr_base)
        gbr.fit(Xtr, ytr)

        yval_pred = gbr.predict(Xval)
        df_va = pd.DataFrame({
            "x_true": yval[:,0], "y_true": yval[:,1],
            "x_pred": yval_pred[:,0], "y_pred": yval_pred[:,1]
        })
        m_va = metrics_from_df(df_va)
        val_rmse = m_va["xy_rmse"]

        print(f"  [CSM tune] val XY-RMSE = {val_rmse:.3f} m (n_triplets={n_rows})")

        records.append({**cfg, "n_triplets": n_rows, "val_xy_rmse": val_rmse})

        if np.isfinite(val_rmse) and val_rmse < best_rmse:
            best_rmse = val_rmse
            best_cfg = cfg.copy()

    df_geom = pd.DataFrame(records)
    df_geom.to_csv(os.path.join(out_subdir, f"{tag_base}_CSM_geometry_grid.csv"), index=False)

    if best_cfg is None:
        print("[CSM tune] No valid configuration found. Using defaults.")
        return None

    # Set best globals
    DBSCAN_EPS       = best_cfg["eps"]
    DBSCAN_MINPTS    = best_cfg["minpts"]
    SPREAD_MAX_M     = best_cfg["spread"]
    TIME_TOLERANCE_S = best_cfg["time_tol"]

    with open(os.path.join(out_subdir, f"{tag_base}_CSM_geometry_best.txt"), "w") as f:
        f.write("Best CSM geometry configuration:\n")
        for k, v in best_cfg.items():
            f.write(f"{k}: {v}\n")
        f.write(f"\nVal XY-RMSE: {best_rmse:.6f} m\n")

    print(f"[CSM tune] Best config = {best_cfg}  (XY-RMSE={best_rmse:.3f} m)")
    return best_cfg

# =============== GBR hyperparameter optimisation ===============

def tune_gbr_hyperparams(Xtr_raw: np.ndarray,
                         ytr: np.ndarray,
                         Xval_raw: np.ndarray,
                         yval: np.ndarray,
                         tag_base: str,
                         feature_mode: str,
                         out_subdir: str) -> dict:
    """
    Simple manual grid search for GBR hyperparameters using train/val split.
    Selection metric: XY-RMSE on validation.
    Returns a dict with:
        "params"         - best hyperparameter dict
        "imp"            - fitted imputer
        "sca"            - fitted scaler
        "model"          - fitted MultiOutputRegressor(GBR)
        "train_metrics"  - metrics on train
        "val_metrics"    - metrics on val
    Also saves a CSV with all tried configs + their metrics.
    """
    # Grid can be extended if you want more exhaustive search
    param_grid = [
        {"n_estimators": 200, "learning_rate": 0.05, "max_depth": 2, "subsample": 0.8},
        {"n_estimators": 300, "learning_rate": 0.05, "max_depth": 3, "subsample": 0.8},
        {"n_estimators": 400, "learning_rate": 0.10, "max_depth": 3, "subsample": 0.9},
        {"n_estimators": 600, "learning_rate": 0.05, "max_depth": 4, "subsample": 0.9},
        {"n_estimators": 600, "learning_rate": 0.1, "max_depth": 3, "subsample": 0.9},
        {"n_estimators": 600, "learning_rate": 0.15, "max_depth": 4, "subsample": 0.9},
        {"n_estimators": 800, "learning_rate": 0.05, "max_depth": 1, "subsample": 0.9},
        {"n_estimators": 800, "learning_rate": 0.1, "max_depth": 3, "subsample": 0.9},
        {"n_estimators": 1000, "learning_rate": 0.1, "max_depth": 3, "subsample": 0.9},
        {"n_estimators": 1000, "learning_rate": 0.05, "max_depth": 4, "subsample": 0.9},
        {"n_estimators": 1500, "learning_rate": 0.05, "max_depth": 4, "subsample": 0.9},
        {"n_estimators": 1500, "learning_rate": 0.1, "max_depth": 4, "subsample": 0.9},
        {"n_estimators": 2000, "learning_rate": 0.1, "max_depth": 4, "subsample": 0.9},
        {"n_estimators": 2500, "learning_rate": 0.1, "max_depth": 4, "subsample": 0.9},
        {"n_estimators": 3000, "learning_rate": 0.1, "max_depth": 4, "subsample": 0.9},


    ]

    results = []
    best_cfg = None
    best_val_rmse = float("inf")

    for i, params in enumerate(param_grid):
        print(f"[GBR tune {feature_mode}] config {i+1}/{len(param_grid)}: {params}")

        imp = SimpleImputer(strategy="mean")
        sca = StandardScaler()

        Xtr0 = imp.fit_transform(Xtr_raw)
        Xval0 = imp.transform(Xval_raw)
        Xtr = sca.fit_transform(Xtr0)
        Xval = sca.transform(Xval0)

        gbr_base = GradientBoostingRegressor(
            n_estimators=params["n_estimators"],
            learning_rate=params["learning_rate"],
            max_depth=params["max_depth"],
            subsample=params["subsample"],
            loss="squared_error",
            random_state=SEED
        )
        gbr = MultiOutputRegressor(gbr_base)
        gbr.fit(Xtr, ytr)

        # Train metrics
        ytr_pred = gbr.predict(Xtr)
        df_tr = pd.DataFrame({
            "x_true": ytr[:,0], "y_true": ytr[:,1],
            "x_pred": ytr_pred[:,0], "y_pred": ytr_pred[:,1]
        })
        m_tr = metrics_from_df(df_tr)

        # Val metrics
        yval_pred = gbr.predict(Xval)
        df_va = pd.DataFrame({
            "x_true": yval[:,0], "y_true": yval[:,1],
            "x_pred": yval_pred[:,0], "y_pred": yval_pred[:,1]
        })
        m_va = metrics_from_df(df_va)

        row = {
            "n_estimators": params["n_estimators"],
            "learning_rate": params["learning_rate"],
            "max_depth": params["max_depth"],
            "subsample": params["subsample"],
            "train_xy_rmse": m_tr["xy_rmse"],
            "train_xy_mae":  m_tr["xy_mae"],
            "val_xy_rmse":   m_va["xy_rmse"],
            "val_xy_mae":    m_va["xy_mae"],
        }
        results.append(row)

        if m_va["xy_rmse"] < best_val_rmse:
            best_val_rmse = m_va["xy_rmse"]
            best_cfg = {
                "params": params,
                "imp": imp,
                "sca": sca,
                "model": gbr,
                "train_metrics": m_tr,
                "val_metrics": m_va,
            }

    # Save grid search results
    df_grid = pd.DataFrame(results)
    df_grid.to_csv(os.path.join(
        out_subdir,
        f"{tag_base}_{feature_mode}_GBR_hyperparam_grid.csv"
    ), index=False)

    print(f"[GBR tune {feature_mode}] best val XY-RMSE={best_val_rmse:.3f} with params={best_cfg['params']}")

    # Save best params as a small text file
    best_txt_path = os.path.join(
        out_subdir,
        f"{tag_base}_{feature_mode}_GBR_best_params.txt"
    )
    with open(best_txt_path, "w") as f:
        f.write(f"Best params (feature_mode={feature_mode}):\n")
        for k,v in best_cfg["params"].items():
            f.write(f"{k}: {v}\n")
        f.write(f"\nBest val XY-RMSE: {best_val_rmse:.6f}\n")

    return best_cfg

# =============== Runner for one feature_mode ===============

def run_gbr_for_feature_mode(train_sel: pd.DataFrame, feature_mode: str, tag_base: str):
    """
    Train/eval GBR (squared error) for one feature_mode.
    Steps:
      - Build X,y from selected triplets.
      - Train/val split.
      - Hyperparameter tuning (manual grid search) using val XY-RMSE.
      - Save:
          * hyperparameter grid results
          * best-GBR train/val metrics CSV + bar plot
          * best hyperparameters (TXT)
          * test metrics CSV (all + cc2-only)
          * predictions & plots for TEST_SUFFIXES
    """
    ensure_dir(OUT_DIR)
    out_subdir = os.path.join(OUT_DIR, f"{tag_base}_{feature_mode}_GBR")
    ensure_dir(out_subdir)

    # === Build X,y
    X_raw, y, feat_cols = build_Xy_from_triplets(train_sel, feature_mode)
    if len(X_raw)==0:
        raise RuntimeError(f"No training rows for feature_mode={feature_mode}")

    # shared train/val split
    Xtr_raw, Xval_raw, ytr, yval = train_test_split(X_raw, y, test_size=VAL_SPLIT, random_state=SEED)

    # === Hyperparameter optimisation for GBR
    best_cfg = tune_gbr_hyperparams(
        Xtr_raw, ytr, Xval_raw, yval,
        tag_base=tag_base,
        feature_mode=feature_mode,
        out_subdir=out_subdir
    )

    imp_gb = best_cfg["imp"]
    sca_gb = best_cfg["sca"]
    gbr     = best_cfg["model"]
    m_tr    = best_cfg["train_metrics"]
    m_va    = best_cfg["val_metrics"]

    def gbr_prep(X):
        X0 = imp_gb.transform(X)
        return sca_gb.transform(X0)

    gbr_predictor = {
        "name": "GBR",
        "prep": gbr_prep,
        "pred": lambda Xs: gbr.predict(Xs)
    }

    # === Train/val metrics (from best config)
    ytr_pred = gbr_predictor["pred"](gbr_predictor["prep"](Xtr_raw))
    yval_pred = gbr_predictor["pred"](gbr_predictor["prep"](Xval_raw))

    df_tr = pd.DataFrame({
        "x_true": ytr[:,0], "y_true": ytr[:,1],
        "x_pred": ytr_pred[:,0], "y_pred": ytr_pred[:,1]
    })
    df_va = pd.DataFrame({
        "x_true": yval[:,0], "y_true": yval[:,1],
        "x_pred": yval_pred[:,0], "y_pred": yval_pred[:,1]
    })

    # re-compute metrics for consistency (or use m_tr/m_va directly)
    m_tr = metrics_from_df(df_tr)
    m_va = metrics_from_df(df_va)

    df_tv = pd.DataFrame([
        {"subset": "train", **m_tr},
        {"subset": "val",   **m_va}
    ])
    df_tv.to_csv(os.path.join(out_subdir,
        f"{tag_base}_{feature_mode}_GBR_train_val_metrics.csv"), index=False)

    # bar plot: train vs val XY RMSE & MAE
    rmse_vals = [m_tr["xy_rmse"], m_va["xy_rmse"]]
    mae_vals  = [m_tr["xy_mae"],  m_va["xy_mae"]]
    labels    = ["train", "val"]

    x = np.arange(len(labels))
    width = 0.35

    plt.figure(figsize=(6,4))
    plt.bar(x - width/2, rmse_vals, width, label="XY RMSE")
    plt.bar(x + width/2, mae_vals,  width, label="XY MAE")
    plt.xticks(x, labels)
    plt.ylabel("Error [m]")
    plt.title(f"{tag_base} ({feature_mode}) — GBR train/val metrics (tuned)")
    plt.legend()
    plt.tight_layout()
    savefig(os.path.join(out_subdir,
        f"{tag_base}_{feature_mode}_GBR_train_val_bar.png"))

    # === Evaluate on TEST suffixes with the tuned model
    df_test = eval_model_on_suffixes_gbr(gbr_predictor, build_Xy_from_triplets,
                                         TEST_SUFFIXES, feature_mode,
                                         tag=tag_base, out_subdir=out_subdir)

    metrics_rows=[]
    if not df_test.empty:
        m_all = metrics_from_df(df_test)
        metrics_rows.append({"feature_mode":feature_mode, "model":"GBR", **m_all})
        df_cc2 = df_test[df_test["suffix"]=="cc2"]
        if not df_cc2.empty:
            m_cc2 = metrics_from_df(df_cc2)
            metrics_rows.append({"feature_mode":feature_mode, "model":"GBR_cc2_only", **m_cc2})

    if metrics_rows:
        df_metrics = pd.DataFrame(metrics_rows)
        df_metrics.to_csv(os.path.join(out_subdir,
            f"{tag_base}_{feature_mode}_GBR_test_metrics.csv"), index=False)

# =============== Main ===============

def main():
    set_seed(SEED)
    ensure_dir(OUT_DIR)
    print(f"[config] device={DEVICE}")
    print("[info] Triplet selector = min-dmid (geometric heuristic)")
    print("[info] Localization model = GBR (squared_error) with hyperparameter tuning")

    # 0) Tune CSM geometry (DBSCAN + gates) using angle-mode + GBR baseline
    print("\n[step] Tuning CSM geometry (DBSCAN_EPS, MINPTS, SPREAD_MAX_M, TIME_TOLERANCE_S)...")
    best_geom = tune_csm_geometry(TRAIN_SUFFIXES, tag_base="CSM")
    if best_geom is not None:
        print(f"[info] Using tuned CSM geometry: {best_geom}")
    else:
        print("[info] Using default CSM geometry (tuning failed or skipped).")

    # 1) Build selected triplets for TRAIN using final CSM geometry
    print("\n[step] Building train-selected triplets with final CSM geometry...")
    sel_rows=[]
    for sfx in TRAIN_SUFFIXES:
        sel = select_triplets_for_suffix(sfx)
        if sel.empty:
            print(f"[train sel] {sfx}: 0 rows"); continue
        sel_rows.append(sel)
        print(f"[train sel] {sfx}: {len(sel)} rows")
    if not sel_rows:
        print("[abort] no selected triplets on TRAIN."); return
    train_sel = pd.concat(sel_rows, ignore_index=True)
    train_sel.to_csv(os.path.join(OUT_DIR, "train_selected_triplets_min_dmid.csv"), index=False)

    # 2) Run GBR for BOTH feature modes (angle & range)
    for FEATURE_MODE in ["angle","range"]:
        print(f"\n=== Running GBR for FEATURE_MODE={FEATURE_MODE} ===")
        run_gbr_for_feature_mode(train_sel, FEATURE_MODE, tag_base="CSM")

if __name__=="__main__":
    main()


[config] device=cuda
[info] Triplet selector = min-dmid (geometric heuristic)
[info] Localization model = GBR (squared_error) with hyperparameter tuning

[step] Tuning CSM geometry (DBSCAN_EPS, MINPTS, SPREAD_MAX_M, TIME_TOLERANCE_S)...

[CSM tune] geometry config 1/7: {'eps': 0.15, 'minpts': 3, 'spread': 0.25, 'time_tol': 0.3}
[CSM] rr1: 44 rows (current geometry)
[CSM] rr2: 30 rows (current geometry)
[CSM] rr3: 49 rows (current geometry)
[CSM] rr4: 35 rows (current geometry)
[CSM] rr5: 38 rows (current geometry)
[CSM] rr6: 35 rows (current geometry)
[CSM] rr7: 29 rows (current geometry)
[CSM] rr8: 49 rows (current geometry)
[CSM] rr9: 54 rows (current geometry)
[CSM] rr12: 40 rows (current geometry)
[CSM] rr11: 40 rows (current geometry)
[CSM] cc3: 35 rows (current geometry)
[CSM] cc5: 47 rows (current geometry)
[CSM] cc6: 38 rows (current geometry)
[CSM] cc7: 28 rows (current geometry)
[CSM] cc8: 39 rows (current geometry)
[CSM] pcc3: 85 rows (current geometry)
[CSM] prr1: 38 rows (

In [2]:
# -*- coding: utf-8 -*-
"""
CSM triplet selection (min-dmid heuristic) + GBR regressor
---------------------------------------------------------
- Triplet selection: deterministic geometric "scorer":
    * Among all candidates per cluster, choose the one whose midpoint
      is closest to the DBSCAN cluster centroid (min dmid).
- Feature switch: angle/range (plus inten, snr, noise per radar).
- Localization model: GradientBoostingRegressor (squared_error), wrapped
  with MultiOutputRegressor for (x, y).
- Hyperparameter optimisation:
    * CSM geometry (DBSCAN_EPS, DBSCAN_MINPTS, SPREAD_MAX_M, TIME_TOLERANCE_S)
      via angle-mode + fixed GBR baseline.
    * GBR hyperparameters via manual grid search per feature_mode (angle/range).
- Saves:
    * Selected training triplets (CSV) for final tuned CSM.
    * CSM geometry grid + best config (CSV/TXT).
    * Train/val metrics for best GBR (CSV & bar plots).
    * GBR hyperparameter grid per feature_mode (CSV).
    * Test metrics (CSV; incl. cc2-only).
    * Test scatter + error histogram for each feature_mode.
"""

import os, glob, math, random
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import DBSCAN
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor

import torch

plt.rcParams.update({
    "figure.dpi": 120, "axes.grid": True, "grid.linestyle": "--",
    "pdf.fonttype": 42, "ps.fonttype": 42
})

# ======================
# === Configuration  ===
# ======================

BASE_PATH = "/home/charithag/master thesis/practise/18Oct/corrected"

# Sessions
TRAIN_SUFFIXES = [
    "rr1","rr2","rr3","rr4","rr5","rr6","rr7","rr8","rr9","rr12","rr11",
    "cc3","cc5","cc6","cc7","cc8","pcc3","prr1","prr2","prr3","prr4",
    "pcc1","pcc2","cc11","cc12","cc13","cc10","cc1","cc2"
]
TEST_SUFFIXES  = ["cc2"]  # explicit cc2 test plots

# Radar roles
FUSION_RADARS   = ["rpi4","rpi1","rpi3"]
REFERENCE_RADARS= ["radar1","rpi2"]  # kept for completeness (not used here)

# Anchors + yaws (deg CCW+)
radar_positions: Dict[str, Tuple[float, float]] = {
    "rpi4": (0.0, 0.0),
    "rpi1": (0.07, 0.7),
    "rpi3": (-0.04, -0.75),
    "rpi5": (2.3, 0.4),
    "rpi6": (2.8, 1.7),
}
sides = {r: {"angle": 0.0} for r in radar_positions.keys()}

# File column mapping (ti_mmwave-style, 0-based)
CSV_X_COL        = 3
CSV_Y_COL        = 4
CSV_RANGE_COL    = 6
CSV_DOPPLER_COL  = 8
CSV_ANGLE_COL    = 9          # (1-based col10)
CSV_INTEN_COL    = 10         # (1-based col11)
CSV_SNR_COL      = 11         # (1-based col12)
CSV_NOISE_COL    = 12         # (1-based col13)
CSV_TIME_COL     = -1

# Pipeline knobs (initial defaults; may be tuned)
BIN_SECONDS      = 0.20
DBSCAN_EPS       = 0.30
DBSCAN_MINPTS    = 3
TIME_TOLERANCE_S = 0.35
SPREAD_MAX_M     = 0.30
K_NEAREST_PER_RADAR = 3
REMOVE_DOPPLER_EQ: Optional[float] = 8.0

# Training knobs (GBR)
SEED        = 1337
VAL_SPLIT   = 0.15

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Output
OUT_DIR = "ML results"  # (user requested this exact folder name)

# ======================
# === Utilities      ===
# ======================

def ensure_dir(p): os.makedirs(p, exist_ok=True)
def savefig(path): ensure_dir(os.path.dirname(path)); plt.savefig(path, dpi=300, bbox_inches="tight"); plt.close()

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def _has_header(path: str) -> bool:
    try:
        with open(path,"r",errors="ignore") as f:
            toks = (f.readline().strip()).split(",")
        for t in toks:
            try: float(t)
            except ValueError: return True
        return False
    except: return False

def _read_any(path: str, header: Optional[int]) -> pd.DataFrame:
    try:    return pd.read_csv(path, header=header, sep=None, engine="python")
    except: return pd.read_csv(path, header=header, delim_whitespace=True)

def _get(df, spec):
    if isinstance(spec,int):
        idx = spec if spec>=0 else df.shape[1]+spec
        return df.iloc[:,idx]
    if isinstance(spec,str):
        if spec in df.columns: return df[spec]
        low = {c.lower():c for c in df.columns}
        if spec.lower() in low: return df[low[spec.lower()]]
    raise KeyError(spec)

def discover_files_for_radar(base: str, radar: str, suffix: str)->List[str]:
    exact = os.path.join(base, radar, suffix)
    if os.path.isfile(exact): return [exact]
    if os.path.isdir(exact):
        files = glob.glob(os.path.join(exact,"**","*"), recursive=True)
        return sorted([f for f in files if os.path.isfile(f)])
    root = os.path.join(base, radar)
    if os.path.isdir(root):
        candidates = glob.glob(os.path.join(root,"**","*"), recursive=True)
        hits = [f for f in candidates if os.path.isfile(f) and os.path.basename(f)==suffix]
        return sorted(hits)
    return []

def load_one_file(path: str) -> pd.DataFrame:
    header = 0 if _has_header(path) else None
    df = _read_any(path, header)
    out = pd.DataFrame({
        "x":        pd.to_numeric(_get(df, CSV_X_COL), errors="coerce"),
        "y":        pd.to_numeric(_get(df, CSV_Y_COL), errors="coerce"),
        "range":    pd.to_numeric(_get(df, CSV_RANGE_COL), errors="coerce"),
        "angle":    pd.to_numeric(_get(df, CSV_ANGLE_COL), errors="coerce"),
        "inten":    pd.to_numeric(_get(df, CSV_INTEN_COL), errors="coerce"),
        "snr":      pd.to_numeric(_get(df, CSV_SNR_COL), errors="coerce"),
        "noise":    pd.to_numeric(_get(df, CSV_NOISE_COL), errors="coerce"),
    })
    try: out["doppler"] = pd.to_numeric(_get(df, CSV_DOPPLER_COL), errors="coerce")
    except: out["doppler"] = np.nan
    try: out["timestamp"] = _get(df, CSV_TIME_COL)
    except: out["timestamp"] = np.nan
    return out.dropna(subset=["x","y"]).reset_index(drop=True)

def load_radar_suffix(base: str, radar: str, suffix: str) -> pd.DataFrame:
    files = discover_files_for_radar(base, radar, suffix)
    if not files:
        return pd.DataFrame(columns=["x","y","range","angle","inten","snr","noise","timestamp","radar"])
    parts=[]
    for f in files:
        try:
            df = load_one_file(f)
            if REMOVE_DOPPLER_EQ is not None and "doppler" in df.columns:
                df = df[~np.isclose(df["doppler"].astype(float), REMOVE_DOPPLER_EQ)]
            df["radar"]=radar
            parts.append(df)
        except Exception as e:
            print(f"[warn] load {f}: {e}")
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

def rot2d(theta):
    c,s=math.cos(theta), math.sin(theta)
    return np.array([[c,-s],[s,c]],float)

def transform_local_to_world(df: pd.DataFrame, yaw_deg: float, tx: float, ty: float)->pd.DataFrame:
    R = rot2d(math.radians(yaw_deg))
    xy = df[["x","y"]].to_numpy(float) @ R.T
    out = df.copy()
    out["xw"]=xy[:,0]+tx; out["yw"]=xy[:,1]+ty
    return out

def load_transform_suffix_all_radars(sfx: str) -> pd.DataFrame:
    parts=[]
    for r,(tx,ty) in radar_positions.items():
        d = load_radar_suffix(BASE_PATH, r, sfx)
        if d.empty: continue
        yaw = float(sides.get(r,{}).get("angle",0.0))
        d = transform_local_to_world(d, yaw, tx, ty)
        parts.append(d)
    if not parts: return pd.DataFrame()
    df = pd.concat(parts, ignore_index=True)
    return df[["x","y","range","angle","inten","snr","noise","timestamp","radar","xw","yw"]]

def to_seconds(s: pd.Series)->pd.Series:
    a = s.astype(str).str.extract(r'([0-9]+(?:\.[0-9]+)?)')[0]
    return pd.to_numeric(a, errors="coerce")

# =============== Binning + Clustering ===============

def bin_time(df: pd.DataFrame, bin_s: float)->pd.DataFrame:
    df = df.dropna(subset=["timestamp"]).copy()
    df["t"] = to_seconds(df["timestamp"])
    df = df.dropna(subset=["t"])
    df["tbin"] = np.floor(df["t"]/bin_s).astype(int)
    return df

def cluster_bin(df_bin: pd.DataFrame)->pd.DataFrame:
    if df_bin.empty: return pd.DataFrame()
    X = df_bin[["xw","yw"]].to_numpy(float)
    labels = DBSCAN(eps=DBSCAN_EPS, min_samples=DBSCAN_MINPTS).fit_predict(X)
    out = df_bin.copy()
    out["cluster"]=labels
    return out

def compute_centroids(df_bin: pd.DataFrame)->Dict[int, Tuple[float,float]]:
    cents={}
    for c, d in df_bin.groupby("cluster"):
        if c==-1: continue
        cents[c]=(float(d["xw"].mean()), float(d["yw"].mean()))
    return cents

# =============== Candidate triplets per cluster ===============

def nearest_k_to_centroid(d: pd.DataFrame, centroid: Tuple[float,float], k:int)->pd.DataFrame:
    dx = d["xw"].to_numpy()-centroid[0]
    dy = d["yw"].to_numpy()-centroid[1]
    dists = np.hypot(dx,dy)
    idx = np.argsort(dists)[:min(k,len(d))]
    return d.iloc[idx].copy()

def build_candidates_for_cluster(d_cluster: pd.DataFrame,
                                 centroid: Tuple[float,float],
                                 fusion_radars: List[str],
                                 k_each:int=K_NEAREST_PER_RADAR)->List[dict]:
    per_radar={}
    for r in fusion_radars:
        dr = d_cluster[d_cluster["radar"]==r]
        if dr.empty:
            return []  # must have all fusion radars
        kn = nearest_k_to_centroid(dr, centroid, k_each).reset_index(drop=True)
        per_radar[r]=kn

    R1,R2,R3 = fusion_radars[:3]
    cands=[]
    n1, n2, n3 = len(per_radar[R1]), len(per_radar[R2]), len(per_radar[R3])
    for ia in range(n1):
        a = per_radar[R1].iloc[ia]
        for ib in range(n2):
            b = per_radar[R2].iloc[ib]
            for ic in range(n3):
                c = per_radar[R3].iloc[ic]
                # time gate
                t_ok = (abs(a["t"]-b["t"])<=TIME_TOLERANCE_S and
                        abs(a["t"]-c["t"])<=TIME_TOLERANCE_S)
                if not t_ok:
                    continue
                # spread gate
                P = np.array([[a["xw"],a["yw"]],
                              [b["xw"],b["yw"]],
                              [c["xw"],c["yw"]]], float)
                d01 = np.hypot(*(P[0]-P[1])); d02 = np.hypot(*(P[0]-P[2])); d12 = np.hypot(*(P[1]-P[2]))
                spread = max(d01,d02,d12)
                if spread>SPREAD_MAX_M:
                    continue
                cands.append({
                    "rows": (ia, ib, ic),
                    "points": P,                 # [3,2] world coords
                    "times": (float(a["t"]), float(b["t"]), float(c["t"])),
                    "centroid": (float(P[:,0].mean()), float(P[:,1].mean()))
                })
    return cands

# =============== Triplet features (for min-dmid) ===============

def triplet_features_for_scoring(cand: dict, cluster_centroid: Tuple[float,float])->np.ndarray:
    """
    z = [d01, d02, d12, spread, dmid, |t0-t1|, |t0-t2|, |t1-t2|]
    We use index 4 (dmid) to implement the min-dmid selector.
    """
    P = cand["points"]; t0,t1,t2 = cand["times"]
    d01 = np.hypot(*(P[0]-P[1])); d02 = np.hypot(*(P[0]-P[2])); d12 = np.hypot(*(P[1]-P[2]))
    spread = max(d01,d02,d12)
    mid = P.mean(axis=0)
    dmid = np.hypot(*(mid - np.array(cluster_centroid)))
    dt = np.array([abs(t0-t1), abs(t0-t2), abs(t1-t2)], float)
    z = np.array([d01,d02,d12,spread,dmid, dt[0],dt[1],dt[2]], float)
    return z

# =============== Selection using min-dmid heuristic ===============

def select_triplets_for_suffix(sfx: str)->pd.DataFrame:
    """
    Deterministic triplet selection for a suffix:
    - Time-bin + DBSCAN per bin.
    - For each cluster:
        * Build candidate triplets (K-nearest per radar, time/spread gates).
        * For each candidate, compute features z.
        * Select candidate with minimal dmid (midpoint-to-cluster-centroid distance).
    Uses global CSM geometry knobs: BIN_SECONDS, DBSCAN_EPS, DBSCAN_MINPTS,
    SPREAD_MAX_M, TIME_TOLERANCE_S, K_NEAREST_PER_RADAR.
    """
    df = load_transform_suffix_all_radars(sfx)
    if df.empty: return pd.DataFrame()
    df = bin_time(df, BIN_SECONDS)
    rows=[]
    for tbin, d_bin in df.groupby("tbin"):
        dc = cluster_bin(d_bin)
        if dc.empty:
            continue
        cents = compute_centroids(dc)
        if not cents: continue
        for cid, d_cluster in dc.groupby("cluster"):
            if cid==-1: continue
            centroid = cents[cid]
            cands = build_candidates_for_cluster(d_cluster, centroid, FUSION_RADARS, K_NEAREST_PER_RADAR)
            if len(cands)==0:
                continue

            # --- min-dmid selection ---
            dmid_values = []
            for c in cands:
                z = triplet_features_for_scoring(c, centroid)
                dmid_values.append(z[4])  # index 4 = dmid
            best_idx = int(np.argmin(dmid_values))
            best = cands[best_idx]
            # ----------------------------

            # rebuild per_radar table to pick actual rows by index
            per_radar={}
            for r in FUSION_RADARS:
                dr = d_cluster[d_cluster["radar"]==r]
                if dr.empty:
                    per_radar = {}
                    break
                per_radar[r]=nearest_k_to_centroid(dr, centroid, K_NEAREST_PER_RADAR).reset_index(drop=True)
            if len(per_radar) < 3:
                continue

            idx1, idx2, idx3 = best["rows"]
            sel = [per_radar[FUSION_RADARS[0]].iloc[idx1],
                   per_radar[FUSION_RADARS[1]].iloc[idx2],
                   per_radar[FUSION_RADARS[2]].iloc[idx3]]

            points = best["points"]; times = best["times"]
            trip = {"suffix": sfx, "tbin": int(tbin), "cluster": int(cid),
                    "x_avg": float(points[:,0].mean()), "y_avg": float(points[:,1].mean()),
                    "t_ref": float(np.mean(times))}
            for r, srow in zip(FUSION_RADARS, sel):
                for k in ["angle","range","inten","snr","noise"]:
                    trip[f"{r}_{k}"]=float(srow[k])
            rows.append(trip)
    return pd.DataFrame(rows)

# =============== Localization model (GBR) ===============

def build_Xy_from_triplets(df_sel: pd.DataFrame, feature_mode:str)->Tuple[np.ndarray,np.ndarray,List[str]]:
    if df_sel.empty: return np.empty((0,12)), np.empty((0,2)), []
    feat_cols=[]
    for r in FUSION_RADARS:
        if feature_mode=="angle":
            feat_cols += [f"{r}_angle", f"{r}_inten", f"{r}_snr", f"{r}_noise"]
        elif feature_mode=="range":
            feat_cols += [f"{r}_range", f"{r}_inten", f"{r}_snr", f"{r}_noise"]
        else:
            raise ValueError("FEATURE_MODE must be 'angle' or 'range'")
    X = df_sel[feat_cols].to_numpy(float)
    y = df_sel[["x_avg","y_avg"]].to_numpy(float)
    return X,y,feat_cols

def metrics_from_df(df: pd.DataFrame)->dict:
    dx = (df["x_pred"]-df["x_true"]).to_numpy(float)
    dy = (df["y_pred"]-df["y_true"]).to_numpy(float)
    err = np.hypot(dx,dy)
    return {
        "x_rmse": float(np.sqrt(np.mean(dx**2))),
        "y_rmse": float(np.sqrt(np.mean(dy**2))),
        "xy_rmse": float(np.sqrt(np.mean(err**2))),
        "xy_mae": float(np.mean(err)),
        "p90": float(np.percentile(err,90)),
        "N": int(len(df))
    }

def scatter_plot(y_true, y_pred, title, path):
    plt.figure(figsize=(5,5))
    plt.scatter(y_true[:,0], y_true[:,1], s=10, label="true")
    plt.scatter(y_pred[:,0], y_pred[:,1], s=10, label="pred")
    plt.axis("equal"); plt.title(title); plt.legend(); plt.tight_layout()
    savefig(path)

def hist_plot(y_true, y_pred, title, path):
    err=np.hypot(y_pred[:,0]-y_true[:,0], y_pred[:,1]-y_true[:,1])
    plt.figure(figsize=(6,4))
    plt.hist(err, bins=40, alpha=0.9)
    plt.xlabel("||error|| [m]"); plt.title(title); plt.tight_layout()
    savefig(path)

def eval_model_on_suffixes_gbr(gbr_predictor, X_builder, suffixes, feature_mode, tag, out_subdir):
    all_rows=[]
    for sfx in suffixes:
        sel = select_triplets_for_suffix(sfx)
        if sel.empty:
            print(f"[test {sfx}] no selected triplets")
            continue
        X_raw, y, feat_cols = X_builder(sel, feature_mode)
        Xs = gbr_predictor["prep"](X_raw)  # preprocessed X
        y_pred = gbr_predictor["pred"](Xs) # np array [N,2]

        df = pd.DataFrame({
            "suffix": sfx,
            "x_true": y[:,0], "y_true": y[:,1],
            "x_pred": y_pred[:,0], "y_pred": y_pred[:,1],
            "tbin": sel["tbin"].values, "cluster": sel["cluster"].values
        })
        all_rows.append(df)

        # save predictions CSV per suffix
        df.to_csv(os.path.join(out_subdir, f"{tag}_{feature_mode}_GBR_{sfx}_predictions.csv"), index=False)

        # plots: scatter + hist
        scatter_plot(y, y_pred,
                     f"Test {sfx} — GBR ({feature_mode})",
                     os.path.join(out_subdir, f"{tag}_{feature_mode}_GBR_{sfx}_test_scatter.png"))
        hist_plot(y, y_pred,
                  f"Test {sfx} — GBR ({feature_mode}) error",
                  os.path.join(out_subdir, f"{tag}_{feature_mode}_GBR_{sfx}_test_errhist.png"))

    return pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()

# =============== Helpers for CSM tuning ===============

def build_train_sel_for_current_csm(train_suffixes: List[str]) -> pd.DataFrame:
    """
    Build concatenated selected triplets for current global CSM configuration.
    """
    sel_rows=[]
    for sfx in train_suffixes:
        sel = select_triplets_for_suffix(sfx)
        if sel.empty:
            print(f"[CSM] {sfx}: 0 rows (current geometry)")
            continue
        sel_rows.append(sel)
        print(f"[CSM] {sfx}: {len(sel)} rows (current geometry)")
    return pd.concat(sel_rows, ignore_index=True) if sel_rows else pd.DataFrame()

def tune_csm_geometry(train_suffixes: List[str], tag_base: str):
    """
    Coarse grid search over CSM geometry parameters:
      - DBSCAN_EPS
      - DBSCAN_MINPTS
      - SPREAD_MAX_M
      - TIME_TOLERANCE_S

    Scoring:
      - Build triplets for all TRAIN_SUFFIXES.
      - Use feature_mode="angle".
      - Train a fixed GBR baseline.
      - Metric: XY-RMSE on validation.
    """
    # Must be declared BEFORE any assignment
    global DBSCAN_EPS, DBSCAN_MINPTS, SPREAD_MAX_M, TIME_TOLERANCE_S

    ensure_dir(OUT_DIR)
    out_subdir = os.path.join(OUT_DIR, f"{tag_base}_CSM_geometry_tuning")
    ensure_dir(out_subdir)

    geometry_grid = [
        {"eps": 0.15, "minpts": 3, "spread": 0.25, "time_tol": 0.30},
        {"eps": 0.20, "minpts": 3, "spread": 0.25, "time_tol": 0.30},     
        {"eps": 0.25, "minpts": 3, "spread": 0.25, "time_tol": 0.30},
        {"eps": 0.30, "minpts": 3, "spread": 0.30, "time_tol": 0.35},
        {"eps": 0.35, "minpts": 3, "spread": 0.35, "time_tol": 0.40},
        {"eps": 0.30, "minpts": 4, "spread": 0.30, "time_tol": 0.35},
        {"eps": 0.35, "minpts": 4, "spread": 0.35, "time_tol": 0.40},
    ]

    records=[]
    best_cfg=None
    best_rmse=float("inf")

    for i, cfg in enumerate(geometry_grid):
        print(f"\n[CSM tune] geometry config {i+1}/{len(geometry_grid)}: {cfg}")

        # Update global knobs
        DBSCAN_EPS       = cfg["eps"]
        DBSCAN_MINPTS    = cfg["minpts"]
        SPREAD_MAX_M     = cfg["spread"]
        TIME_TOLERANCE_S = cfg["time_tol"]

        train_sel = build_train_sel_for_current_csm(train_suffixes)
        n_rows = len(train_sel)

        if n_rows < 200:
            print(f"  [skip] only {n_rows} triplets; skipping config.")
            records.append({**cfg, "n_triplets": n_rows, "val_xy_rmse": np.nan})
            continue

        X_raw, y, _ = build_Xy_from_triplets(train_sel, "angle")
        Xtr_raw, Xval_raw, ytr, yval = train_test_split(
            X_raw, y, test_size=VAL_SPLIT, random_state=SEED
        )

        imp = SimpleImputer(strategy="mean")
        sca = StandardScaler()
        Xtr = sca.fit_transform(imp.fit_transform(Xtr_raw))
        Xval = sca.transform(imp.transform(Xval_raw))

        gbr_base = GradientBoostingRegressor(
            n_estimators=300,
            learning_rate=0.08,
            max_depth=3,
            subsample=0.9,
            loss="huber",
            random_state=SEED
        )
        gbr = MultiOutputRegressor(gbr_base)
        gbr.fit(Xtr, ytr)

        yval_pred = gbr.predict(Xval)
        df_va = pd.DataFrame({
            "x_true": yval[:,0], "y_true": yval[:,1],
            "x_pred": yval_pred[:,0], "y_pred": yval_pred[:,1]
        })
        m_va = metrics_from_df(df_va)
        val_rmse = m_va["xy_rmse"]

        print(f"  [CSM tune] val XY-RMSE = {val_rmse:.3f} m (n_triplets={n_rows})")

        records.append({**cfg, "n_triplets": n_rows, "val_xy_rmse": val_rmse})

        if np.isfinite(val_rmse) and val_rmse < best_rmse:
            best_rmse = val_rmse
            best_cfg = cfg.copy()

    df_geom = pd.DataFrame(records)
    df_geom.to_csv(os.path.join(out_subdir, f"{tag_base}_CSM_geometry_grid.csv"), index=False)

    if best_cfg is None:
        print("[CSM tune] No valid configuration found. Using defaults.")
        return None

    # Set best globals
    DBSCAN_EPS       = best_cfg["eps"]
    DBSCAN_MINPTS    = best_cfg["minpts"]
    SPREAD_MAX_M     = best_cfg["spread"]
    TIME_TOLERANCE_S = best_cfg["time_tol"]

    with open(os.path.join(out_subdir, f"{tag_base}_CSM_geometry_best.txt"), "w") as f:
        f.write("Best CSM geometry configuration:\n")
        for k, v in best_cfg.items():
            f.write(f"{k}: {v}\n")
        f.write(f"\nVal XY-RMSE: {best_rmse:.6f} m\n")

    print(f"[CSM tune] Best config = {best_cfg}  (XY-RMSE={best_rmse:.3f} m)")
    return best_cfg

# =============== GBR hyperparameter optimisation ===============

def tune_gbr_hyperparams(Xtr_raw: np.ndarray,
                         ytr: np.ndarray,
                         Xval_raw: np.ndarray,
                         yval: np.ndarray,
                         tag_base: str,
                         feature_mode: str,
                         out_subdir: str) -> dict:
    """
    Simple manual grid search for GBR hyperparameters using train/val split.
    Selection metric: XY-RMSE on validation.
    Returns a dict with:
        "params"         - best hyperparameter dict
        "imp"            - fitted imputer
        "sca"            - fitted scaler
        "model"          - fitted MultiOutputRegressor(GBR)
        "train_metrics"  - metrics on train
        "val_metrics"    - metrics on val
    Also saves a CSV with all tried configs + their metrics.
    """
    # Grid can be extended if you want more exhaustive search
    param_grid = [
        {"n_estimators": 200, "learning_rate": 0.05, "max_depth": 2, "subsample": 0.8},
        {"n_estimators": 300, "learning_rate": 0.05, "max_depth": 3, "subsample": 0.8},
        {"n_estimators": 400, "learning_rate": 0.10, "max_depth": 3, "subsample": 0.9},
        {"n_estimators": 600, "learning_rate": 0.05, "max_depth": 4, "subsample": 0.9},
        {"n_estimators": 600, "learning_rate": 0.1, "max_depth": 3, "subsample": 0.9},
        {"n_estimators": 600, "learning_rate": 0.15, "max_depth": 4, "subsample": 0.9},
        {"n_estimators": 800, "learning_rate": 0.05, "max_depth": 1, "subsample": 0.9},
        {"n_estimators": 800, "learning_rate": 0.1, "max_depth": 3, "subsample": 0.9},
        {"n_estimators": 1000, "learning_rate": 0.1, "max_depth": 3, "subsample": 0.9},
        {"n_estimators": 1000, "learning_rate": 0.05, "max_depth": 4, "subsample": 0.9},
        {"n_estimators": 1500, "learning_rate": 0.05, "max_depth": 4, "subsample": 0.9},
        {"n_estimators": 1500, "learning_rate": 0.1, "max_depth": 4, "subsample": 0.9},
        {"n_estimators": 2000, "learning_rate": 0.1, "max_depth": 4, "subsample": 0.9},
        {"n_estimators": 2500, "learning_rate": 0.1, "max_depth": 4, "subsample": 0.9},
        {"n_estimators": 3000, "learning_rate": 0.1, "max_depth": 4, "subsample": 0.9},


    ]

    results = []
    best_cfg = None
    best_val_rmse = float("inf")

    for i, params in enumerate(param_grid):
        print(f"[GBR tune {feature_mode}] config {i+1}/{len(param_grid)}: {params}")

        imp = SimpleImputer(strategy="mean")
        sca = StandardScaler()

        Xtr0 = imp.fit_transform(Xtr_raw)
        Xval0 = imp.transform(Xval_raw)
        Xtr = sca.fit_transform(Xtr0)
        Xval = sca.transform(Xval0)

        gbr_base = GradientBoostingRegressor(
            n_estimators=params["n_estimators"],
            learning_rate=params["learning_rate"],
            max_depth=params["max_depth"],
            subsample=params["subsample"],
            loss="squared_error",
            random_state=SEED
        )
        gbr = MultiOutputRegressor(gbr_base)
        gbr.fit(Xtr, ytr)

        # Train metrics
        ytr_pred = gbr.predict(Xtr)
        df_tr = pd.DataFrame({
            "x_true": ytr[:,0], "y_true": ytr[:,1],
            "x_pred": ytr_pred[:,0], "y_pred": ytr_pred[:,1]
        })
        m_tr = metrics_from_df(df_tr)

        # Val metrics
        yval_pred = gbr.predict(Xval)
        df_va = pd.DataFrame({
            "x_true": yval[:,0], "y_true": yval[:,1],
            "x_pred": yval_pred[:,0], "y_pred": yval_pred[:,1]
        })
        m_va = metrics_from_df(df_va)

        row = {
            "n_estimators": params["n_estimators"],
            "learning_rate": params["learning_rate"],
            "max_depth": params["max_depth"],
            "subsample": params["subsample"],
            "train_xy_rmse": m_tr["xy_rmse"],
            "train_xy_mae":  m_tr["xy_mae"],
            "val_xy_rmse":   m_va["xy_rmse"],
            "val_xy_mae":    m_va["xy_mae"],
        }
        results.append(row)

        if m_va["xy_rmse"] < best_val_rmse:
            best_val_rmse = m_va["xy_rmse"]
            best_cfg = {
                "params": params,
                "imp": imp,
                "sca": sca,
                "model": gbr,
                "train_metrics": m_tr,
                "val_metrics": m_va,
            }

    # Save grid search results
    df_grid = pd.DataFrame(results)
    df_grid.to_csv(os.path.join(
        out_subdir,
        f"{tag_base}_{feature_mode}_GBR_hyperparam_grid.csv"
    ), index=False)

    print(f"[GBR tune {feature_mode}] best val XY-RMSE={best_val_rmse:.3f} with params={best_cfg['params']}")

    # Save best params as a small text file
    best_txt_path = os.path.join(
        out_subdir,
        f"{tag_base}_{feature_mode}_GBR_best_params.txt"
    )
    with open(best_txt_path, "w") as f:
        f.write(f"Best params (feature_mode={feature_mode}):\n")
        for k,v in best_cfg["params"].items():
            f.write(f"{k}: {v}\n")
        f.write(f"\nBest val XY-RMSE: {best_val_rmse:.6f}\n")

    return best_cfg

# =============== Runner for one feature_mode ===============

def run_gbr_for_feature_mode(train_sel: pd.DataFrame, feature_mode: str, tag_base: str):
    """
    Train/eval GBR (squared error) for one feature_mode.
    Steps:
      - Build X,y from selected triplets.
      - Train/val split.
      - Hyperparameter tuning (manual grid search) using val XY-RMSE.
      - Save:
          * hyperparameter grid results
          * best-GBR train/val metrics CSV + bar plot
          * best hyperparameters (TXT)
          * test metrics CSV (all + cc2-only)
          * predictions & plots for TEST_SUFFIXES
    """
    ensure_dir(OUT_DIR)
    out_subdir = os.path.join(OUT_DIR, f"{tag_base}_{feature_mode}_GBR")
    ensure_dir(out_subdir)

    # === Build X,y
    X_raw, y, feat_cols = build_Xy_from_triplets(train_sel, feature_mode)
    if len(X_raw)==0:
        raise RuntimeError(f"No training rows for feature_mode={feature_mode}")

    # shared train/val split
    Xtr_raw, Xval_raw, ytr, yval = train_test_split(X_raw, y, test_size=VAL_SPLIT, random_state=SEED)

    # === Hyperparameter optimisation for GBR
    best_cfg = tune_gbr_hyperparams(
        Xtr_raw, ytr, Xval_raw, yval,
        tag_base=tag_base,
        feature_mode=feature_mode,
        out_subdir=out_subdir
    )

    imp_gb = best_cfg["imp"]
    sca_gb = best_cfg["sca"]
    gbr     = best_cfg["model"]
    m_tr    = best_cfg["train_metrics"]
    m_va    = best_cfg["val_metrics"]

    def gbr_prep(X):
        X0 = imp_gb.transform(X)
        return sca_gb.transform(X0)

    gbr_predictor = {
        "name": "GBR",
        "prep": gbr_prep,
        "pred": lambda Xs: gbr.predict(Xs)
    }

    # === Train/val metrics (from best config)
    ytr_pred = gbr_predictor["pred"](gbr_predictor["prep"](Xtr_raw))
    yval_pred = gbr_predictor["pred"](gbr_predictor["prep"](Xval_raw))

    df_tr = pd.DataFrame({
        "x_true": ytr[:,0], "y_true": ytr[:,1],
        "x_pred": ytr_pred[:,0], "y_pred": ytr_pred[:,1]
    })
    df_va = pd.DataFrame({
        "x_true": yval[:,0], "y_true": yval[:,1],
        "x_pred": yval_pred[:,0], "y_pred": yval_pred[:,1]
    })

    # re-compute metrics for consistency (or use m_tr/m_va directly)
    m_tr = metrics_from_df(df_tr)
    m_va = metrics_from_df(df_va)

    df_tv = pd.DataFrame([
        {"subset": "train", **m_tr},
        {"subset": "val",   **m_va}
    ])
    df_tv.to_csv(os.path.join(out_subdir,
        f"{tag_base}_{feature_mode}_GBR_train_val_metrics.csv"), index=False)

    # bar plot: train vs val XY RMSE & MAE
    rmse_vals = [m_tr["xy_rmse"], m_va["xy_rmse"]]
    mae_vals  = [m_tr["xy_mae"],  m_va["xy_mae"]]
    labels    = ["train", "val"]

    x = np.arange(len(labels))
    width = 0.35

    plt.figure(figsize=(6,4))
    plt.bar(x - width/2, rmse_vals, width, label="XY RMSE")
    plt.bar(x + width/2, mae_vals,  width, label="XY MAE")
    plt.xticks(x, labels)
    plt.ylabel("Error [m]")
    plt.title(f"{tag_base} ({feature_mode}) — GBR train/val metrics (tuned)")
    plt.legend()
    plt.tight_layout()
    savefig(os.path.join(out_subdir,
        f"{tag_base}_{feature_mode}_GBR_train_val_bar.png"))

    # === Evaluate on TEST suffixes with the tuned model
    df_test = eval_model_on_suffixes_gbr(gbr_predictor, build_Xy_from_triplets,
                                         TEST_SUFFIXES, feature_mode,
                                         tag=tag_base, out_subdir=out_subdir)

    metrics_rows=[]
    if not df_test.empty:
        m_all = metrics_from_df(df_test)
        metrics_rows.append({"feature_mode":feature_mode, "model":"GBR", **m_all})
        df_cc2 = df_test[df_test["suffix"]=="cc2"]
        if not df_cc2.empty:
            m_cc2 = metrics_from_df(df_cc2)
            metrics_rows.append({"feature_mode":feature_mode, "model":"GBR_cc2_only", **m_cc2})

    if metrics_rows:
        df_metrics = pd.DataFrame(metrics_rows)
        df_metrics.to_csv(os.path.join(out_subdir,
            f"{tag_base}_{feature_mode}_GBR_test_metrics.csv"), index=False)

# =============== Main ===============

def main():
    set_seed(SEED)
    ensure_dir(OUT_DIR)
    print(f"[config] device={DEVICE}")
    print("[info] Triplet selector = min-dmid (geometric heuristic)")
    print("[info] Localization model = GBR (squared_error) with hyperparameter tuning")

    # 0) Tune CSM geometry (DBSCAN + gates) using angle-mode + GBR baseline
    print("\n[step] Tuning CSM geometry (DBSCAN_EPS, MINPTS, SPREAD_MAX_M, TIME_TOLERANCE_S)...")
    best_geom = tune_csm_geometry(TRAIN_SUFFIXES, tag_base="CSM")
    if best_geom is not None:
        print(f"[info] Using tuned CSM geometry: {best_geom}")
    else:
        print("[info] Using default CSM geometry (tuning failed or skipped).")

    # 1) Build selected triplets for TRAIN using final CSM geometry
    print("\n[step] Building train-selected triplets with final CSM geometry...")
    sel_rows=[]
    for sfx in TRAIN_SUFFIXES:
        sel = select_triplets_for_suffix(sfx)
        if sel.empty:
            print(f"[train sel] {sfx}: 0 rows"); continue
        sel_rows.append(sel)
        print(f"[train sel] {sfx}: {len(sel)} rows")
    if not sel_rows:
        print("[abort] no selected triplets on TRAIN."); return
    train_sel = pd.concat(sel_rows, ignore_index=True)
    train_sel.to_csv(os.path.join(OUT_DIR, "train_selected_triplets_min_dmid.csv"), index=False)

    # 2) Run GBR for BOTH feature modes (angle & range)
    for FEATURE_MODE in ["angle","range"]:
        print(f"\n=== Running GBR for FEATURE_MODE={FEATURE_MODE} ===")
        run_gbr_for_feature_mode(train_sel, FEATURE_MODE, tag_base="CSM")

if __name__=="__main__":
    main()


[config] device=cuda
[info] Triplet selector = min-dmid (geometric heuristic)
[info] Localization model = GBR (squared_error) with hyperparameter tuning

[step] Tuning CSM geometry (DBSCAN_EPS, MINPTS, SPREAD_MAX_M, TIME_TOLERANCE_S)...

[CSM tune] geometry config 1/7: {'eps': 0.15, 'minpts': 3, 'spread': 0.25, 'time_tol': 0.3}
[CSM] rr1: 44 rows (current geometry)
[CSM] rr2: 30 rows (current geometry)
[CSM] rr3: 49 rows (current geometry)
[CSM] rr4: 35 rows (current geometry)
[CSM] rr5: 38 rows (current geometry)
[CSM] rr6: 35 rows (current geometry)
[CSM] rr7: 29 rows (current geometry)
[CSM] rr8: 49 rows (current geometry)
[CSM] rr9: 54 rows (current geometry)
[CSM] rr12: 40 rows (current geometry)
[CSM] rr11: 40 rows (current geometry)
[CSM] cc3: 35 rows (current geometry)
[CSM] cc5: 47 rows (current geometry)
[CSM] cc6: 38 rows (current geometry)
[CSM] cc7: 28 rows (current geometry)
[CSM] cc8: 39 rows (current geometry)
[CSM] pcc3: 85 rows (current geometry)
[CSM] prr1: 38 rows (

In [3]:
# -*- coding: utf-8 -*-
"""
CSM triplet selection (min-dmid heuristic) + GBR regressor
---------------------------------------------------------
- Triplet selection: deterministic geometric "scorer":
    * Among all candidates per cluster, choose the one whose midpoint
      is closest to the DBSCAN cluster centroid (min dmid).
- Feature switch: angle/range (plus inten, snr, noise per radar).
- Localization model: GradientBoostingRegressor (squared_error), wrapped
  with MultiOutputRegressor for (x, y).
- Hyperparameter optimisation:
    * CSM geometry (DBSCAN_EPS, DBSCAN_MINPTS, SPREAD_MAX_M, TIME_TOLERANCE_S)
      via angle-mode + fixed GBR baseline.
    * GBR hyperparameters via manual grid search per feature_mode (angle/range).
- Saves:
    * Selected training triplets (CSV) for final tuned CSM.
    * CSM geometry grid + best config (CSV/TXT).
    * Train/val metrics for best GBR (CSV & bar plots).
    * GBR hyperparameter grid per feature_mode (CSV).
    * Test metrics (CSV; incl. cc2-only).
    * Test scatter + error histogram for each feature_mode.
"""

import os, glob, math, random
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import DBSCAN
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor

import torch

plt.rcParams.update({
    "figure.dpi": 120, "axes.grid": True, "grid.linestyle": "--",
    "pdf.fonttype": 42, "ps.fonttype": 42
})

# ======================
# === Configuration  ===
# ======================

BASE_PATH = "/home/charithag/master thesis/practise/18Oct/corrected"

# Sessions
TRAIN_SUFFIXES = [
    "rr1","rr2","rr3","rr4","rr5","rr6","rr7","rr8","rr9","rr12","rr11",
    "cc3","cc5","cc6","cc7","cc8","pcc3","prr1","prr2","prr3","prr4",
    "pcc1","pcc2","cc11","cc12","cc13","cc10","cc1","cc2"
]
TEST_SUFFIXES  = ["cc2"]  # explicit cc2 test plots

# Radar roles
FUSION_RADARS   = ["rpi4","rpi1","rpi3"]
REFERENCE_RADARS= ["radar1","rpi2"]  # kept for completeness (not used here)

# Anchors + yaws (deg CCW+)
radar_positions: Dict[str, Tuple[float, float]] = {
    "rpi4": (0.0, 0.0),
    "rpi1": (0.07, 0.7),
    "rpi3": (-0.04, -0.75),
    "rpi5": (2.3, 0.4),
    "rpi6": (2.8, 1.7),
}
sides = {r: {"angle": 0.0} for r in radar_positions.keys()}

# File column mapping (ti_mmwave-style, 0-based)
CSV_X_COL        = 3
CSV_Y_COL        = 4
CSV_RANGE_COL    = 6
CSV_DOPPLER_COL  = 8
CSV_ANGLE_COL    = 9          # (1-based col10)
CSV_INTEN_COL    = 10         # (1-based col11)
CSV_SNR_COL      = 11         # (1-based col12)
CSV_NOISE_COL    = 12         # (1-based col13)
CSV_TIME_COL     = -1

# Pipeline knobs (initial defaults; may be tuned)
BIN_SECONDS      = 0.20
DBSCAN_EPS       = 0.30
DBSCAN_MINPTS    = 3
TIME_TOLERANCE_S = 0.35
SPREAD_MAX_M     = 0.30
K_NEAREST_PER_RADAR = 3
REMOVE_DOPPLER_EQ: Optional[float] = 8.0

# Training knobs (GBR)
SEED        = 1337
VAL_SPLIT   = 0.15

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Output
OUT_DIR = "ML results"  # (user requested this exact folder name)

# ======================
# === Utilities      ===
# ======================

def ensure_dir(p): os.makedirs(p, exist_ok=True)
def savefig(path): ensure_dir(os.path.dirname(path)); plt.savefig(path, dpi=300, bbox_inches="tight"); plt.close()

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def _has_header(path: str) -> bool:
    try:
        with open(path,"r",errors="ignore") as f:
            toks = (f.readline().strip()).split(",")
        for t in toks:
            try: float(t)
            except ValueError: return True
        return False
    except: return False

def _read_any(path: str, header: Optional[int]) -> pd.DataFrame:
    try:    return pd.read_csv(path, header=header, sep=None, engine="python")
    except: return pd.read_csv(path, header=header, delim_whitespace=True)

def _get(df, spec):
    if isinstance(spec,int):
        idx = spec if spec>=0 else df.shape[1]+spec
        return df.iloc[:,idx]
    if isinstance(spec,str):
        if spec in df.columns: return df[spec]
        low = {c.lower():c for c in df.columns}
        if spec.lower() in low: return df[low[spec.lower()]]
    raise KeyError(spec)

def discover_files_for_radar(base: str, radar: str, suffix: str)->List[str]:
    exact = os.path.join(base, radar, suffix)
    if os.path.isfile(exact): return [exact]
    if os.path.isdir(exact):
        files = glob.glob(os.path.join(exact,"**","*"), recursive=True)
        return sorted([f for f in files if os.path.isfile(f)])
    root = os.path.join(base, radar)
    if os.path.isdir(root):
        candidates = glob.glob(os.path.join(root,"**","*"), recursive=True)
        hits = [f for f in candidates if os.path.isfile(f) and os.path.basename(f)==suffix]
        return sorted(hits)
    return []

def load_one_file(path: str) -> pd.DataFrame:
    header = 0 if _has_header(path) else None
    df = _read_any(path, header)
    out = pd.DataFrame({
        "x":        pd.to_numeric(_get(df, CSV_X_COL), errors="coerce"),
        "y":        pd.to_numeric(_get(df, CSV_Y_COL), errors="coerce"),
        "range":    pd.to_numeric(_get(df, CSV_RANGE_COL), errors="coerce"),
        "angle":    pd.to_numeric(_get(df, CSV_ANGLE_COL), errors="coerce"),
        "inten":    pd.to_numeric(_get(df, CSV_INTEN_COL), errors="coerce"),
        "snr":      pd.to_numeric(_get(df, CSV_SNR_COL), errors="coerce"),
        "noise":    pd.to_numeric(_get(df, CSV_NOISE_COL), errors="coerce"),
    })
    try: out["doppler"] = pd.to_numeric(_get(df, CSV_DOPPLER_COL), errors="coerce")
    except: out["doppler"] = np.nan
    try: out["timestamp"] = _get(df, CSV_TIME_COL)
    except: out["timestamp"] = np.nan
    return out.dropna(subset=["x","y"]).reset_index(drop=True)

def load_radar_suffix(base: str, radar: str, suffix: str) -> pd.DataFrame:
    files = discover_files_for_radar(base, radar, suffix)
    if not files:
        return pd.DataFrame(columns=["x","y","range","angle","inten","snr","noise","timestamp","radar"])
    parts=[]
    for f in files:
        try:
            df = load_one_file(f)
            if REMOVE_DOPPLER_EQ is not None and "doppler" in df.columns:
                df = df[~np.isclose(df["doppler"].astype(float), REMOVE_DOPPLER_EQ)]
            df["radar"]=radar
            parts.append(df)
        except Exception as e:
            print(f"[warn] load {f}: {e}")
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

def rot2d(theta):
    c,s=math.cos(theta), math.sin(theta)
    return np.array([[c,-s],[s,c]],float)

def transform_local_to_world(df: pd.DataFrame, yaw_deg: float, tx: float, ty: float)->pd.DataFrame:
    R = rot2d(math.radians(yaw_deg))
    xy = df[["x","y"]].to_numpy(float) @ R.T
    out = df.copy()
    out["xw"]=xy[:,0]+tx; out["yw"]=xy[:,1]+ty
    return out

def load_transform_suffix_all_radars(sfx: str) -> pd.DataFrame:
    parts=[]
    for r,(tx,ty) in radar_positions.items():
        d = load_radar_suffix(BASE_PATH, r, sfx)
        if d.empty: continue
        yaw = float(sides.get(r,{}).get("angle",0.0))
        d = transform_local_to_world(d, yaw, tx, ty)
        parts.append(d)
    if not parts: return pd.DataFrame()
    df = pd.concat(parts, ignore_index=True)
    return df[["x","y","range","angle","inten","snr","noise","timestamp","radar","xw","yw"]]

def to_seconds(s: pd.Series)->pd.Series:
    a = s.astype(str).str.extract(r'([0-9]+(?:\.[0-9]+)?)')[0]
    return pd.to_numeric(a, errors="coerce")

# =============== Binning + Clustering ===============

def bin_time(df: pd.DataFrame, bin_s: float)->pd.DataFrame:
    df = df.dropna(subset=["timestamp"]).copy()
    df["t"] = to_seconds(df["timestamp"])
    df = df.dropna(subset=["t"])
    df["tbin"] = np.floor(df["t"]/bin_s).astype(int)
    return df

def cluster_bin(df_bin: pd.DataFrame)->pd.DataFrame:
    if df_bin.empty: return pd.DataFrame()
    X = df_bin[["xw","yw"]].to_numpy(float)
    labels = DBSCAN(eps=DBSCAN_EPS, min_samples=DBSCAN_MINPTS).fit_predict(X)
    out = df_bin.copy()
    out["cluster"]=labels
    return out

def compute_centroids(df_bin: pd.DataFrame)->Dict[int, Tuple[float,float]]:
    cents={}
    for c, d in df_bin.groupby("cluster"):
        if c==-1: continue
        cents[c]=(float(d["xw"].mean()), float(d["yw"].mean()))
    return cents

# =============== Candidate triplets per cluster ===============

def nearest_k_to_centroid(d: pd.DataFrame, centroid: Tuple[float,float], k:int)->pd.DataFrame:
    dx = d["xw"].to_numpy()-centroid[0]
    dy = d["yw"].to_numpy()-centroid[1]
    dists = np.hypot(dx,dy)
    idx = np.argsort(dists)[:min(k,len(d))]
    return d.iloc[idx].copy()

def build_candidates_for_cluster(d_cluster: pd.DataFrame,
                                 centroid: Tuple[float,float],
                                 fusion_radars: List[str],
                                 k_each:int=K_NEAREST_PER_RADAR)->List[dict]:
    per_radar={}
    for r in fusion_radars:
        dr = d_cluster[d_cluster["radar"]==r]
        if dr.empty:
            return []  # must have all fusion radars
        kn = nearest_k_to_centroid(dr, centroid, k_each).reset_index(drop=True)
        per_radar[r]=kn

    R1,R2,R3 = fusion_radars[:3]
    cands=[]
    n1, n2, n3 = len(per_radar[R1]), len(per_radar[R2]), len(per_radar[R3])
    for ia in range(n1):
        a = per_radar[R1].iloc[ia]
        for ib in range(n2):
            b = per_radar[R2].iloc[ib]
            for ic in range(n3):
                c = per_radar[R3].iloc[ic]
                # time gate
                t_ok = (abs(a["t"]-b["t"])<=TIME_TOLERANCE_S and
                        abs(a["t"]-c["t"])<=TIME_TOLERANCE_S)
                if not t_ok:
                    continue
                # spread gate
                P = np.array([[a["xw"],a["yw"]],
                              [b["xw"],b["yw"]],
                              [c["xw"],c["yw"]]], float)
                d01 = np.hypot(*(P[0]-P[1])); d02 = np.hypot(*(P[0]-P[2])); d12 = np.hypot(*(P[1]-P[2]))
                spread = max(d01,d02,d12)
                if spread>SPREAD_MAX_M:
                    continue
                cands.append({
                    "rows": (ia, ib, ic),
                    "points": P,                 # [3,2] world coords
                    "times": (float(a["t"]), float(b["t"]), float(c["t"])),
                    "centroid": (float(P[:,0].mean()), float(P[:,1].mean()))
                })
    return cands

# =============== Triplet features (for min-dmid) ===============

def triplet_features_for_scoring(cand: dict, cluster_centroid: Tuple[float,float])->np.ndarray:
    """
    z = [d01, d02, d12, spread, dmid, |t0-t1|, |t0-t2|, |t1-t2|]
    We use index 4 (dmid) to implement the min-dmid selector.
    """
    P = cand["points"]; t0,t1,t2 = cand["times"]
    d01 = np.hypot(*(P[0]-P[1])); d02 = np.hypot(*(P[0]-P[2])); d12 = np.hypot(*(P[1]-P[2]))
    spread = max(d01,d02,d12)
    mid = P.mean(axis=0)
    dmid = np.hypot(*(mid - np.array(cluster_centroid)))
    dt = np.array([abs(t0-t1), abs(t0-t2), abs(t1-t2)], float)
    z = np.array([d01,d02,d12,spread,dmid, dt[0],dt[1],dt[2]], float)
    return z

# =============== Selection using min-dmid heuristic ===============

def select_triplets_for_suffix(sfx: str)->pd.DataFrame:
    """
    Deterministic triplet selection for a suffix:
    - Time-bin + DBSCAN per bin.
    - For each cluster:
        * Build candidate triplets (K-nearest per radar, time/spread gates).
        * For each candidate, compute features z.
        * Select candidate with minimal dmid (midpoint-to-cluster-centroid distance).
    Uses global CSM geometry knobs: BIN_SECONDS, DBSCAN_EPS, DBSCAN_MINPTS,
    SPREAD_MAX_M, TIME_TOLERANCE_S, K_NEAREST_PER_RADAR.
    """
    df = load_transform_suffix_all_radars(sfx)
    if df.empty: return pd.DataFrame()
    df = bin_time(df, BIN_SECONDS)
    rows=[]
    for tbin, d_bin in df.groupby("tbin"):
        dc = cluster_bin(d_bin)
        if dc.empty:
            continue
        cents = compute_centroids(dc)
        if not cents: continue
        for cid, d_cluster in dc.groupby("cluster"):
            if cid==-1: continue
            centroid = cents[cid]
            cands = build_candidates_for_cluster(d_cluster, centroid, FUSION_RADARS, K_NEAREST_PER_RADAR)
            if len(cands)==0:
                continue

            # --- min-dmid selection ---
            dmid_values = []
            for c in cands:
                z = triplet_features_for_scoring(c, centroid)
                dmid_values.append(z[4])  # index 4 = dmid
            best_idx = int(np.argmin(dmid_values))
            best = cands[best_idx]
            # ----------------------------

            # rebuild per_radar table to pick actual rows by index
            per_radar={}
            for r in FUSION_RADARS:
                dr = d_cluster[d_cluster["radar"]==r]
                if dr.empty:
                    per_radar = {}
                    break
                per_radar[r]=nearest_k_to_centroid(dr, centroid, K_NEAREST_PER_RADAR).reset_index(drop=True)
            if len(per_radar) < 3:
                continue

            idx1, idx2, idx3 = best["rows"]
            sel = [per_radar[FUSION_RADARS[0]].iloc[idx1],
                   per_radar[FUSION_RADARS[1]].iloc[idx2],
                   per_radar[FUSION_RADARS[2]].iloc[idx3]]

            points = best["points"]; times = best["times"]
            trip = {"suffix": sfx, "tbin": int(tbin), "cluster": int(cid),
                    "x_avg": float(points[:,0].mean()), "y_avg": float(points[:,1].mean()),
                    "t_ref": float(np.mean(times))}
            for r, srow in zip(FUSION_RADARS, sel):
                for k in ["angle","range","inten","snr","noise"]:
                    trip[f"{r}_{k}"]=float(srow[k])
            rows.append(trip)
    return pd.DataFrame(rows)

# =============== Localization model (GBR) ===============

def build_Xy_from_triplets(df_sel: pd.DataFrame, feature_mode:str)->Tuple[np.ndarray,np.ndarray,List[str]]:
    if df_sel.empty: return np.empty((0,12)), np.empty((0,2)), []
    feat_cols=[]
    for r in FUSION_RADARS:
        if feature_mode=="angle":
            feat_cols += [f"{r}_angle", f"{r}_inten", f"{r}_snr", f"{r}_noise"]
        elif feature_mode=="range":
            feat_cols += [f"{r}_range", f"{r}_inten", f"{r}_snr", f"{r}_noise"]
        else:
            raise ValueError("FEATURE_MODE must be 'angle' or 'range'")
    X = df_sel[feat_cols].to_numpy(float)
    y = df_sel[["x_avg","y_avg"]].to_numpy(float)
    return X,y,feat_cols

def metrics_from_df(df: pd.DataFrame)->dict:
    dx = (df["x_pred"]-df["x_true"]).to_numpy(float)
    dy = (df["y_pred"]-df["y_true"]).to_numpy(float)
    err = np.hypot(dx,dy)
    return {
        "x_rmse": float(np.sqrt(np.mean(dx**2))),
        "y_rmse": float(np.sqrt(np.mean(dy**2))),
        "xy_rmse": float(np.sqrt(np.mean(err**2))),
        "xy_mae": float(np.mean(err)),
        "p90": float(np.percentile(err,90)),
        "N": int(len(df))
    }

def scatter_plot(y_true, y_pred, title, path):
    plt.figure(figsize=(5,5))
    plt.scatter(y_true[:,0], y_true[:,1], s=10, label="true")
    plt.scatter(y_pred[:,0], y_pred[:,1], s=10, label="pred")
    plt.axis("equal"); plt.title(title); plt.legend(); plt.tight_layout()
    savefig(path)

def hist_plot(y_true, y_pred, title, path):
    err=np.hypot(y_pred[:,0]-y_true[:,0], y_pred[:,1]-y_true[:,1])
    plt.figure(figsize=(6,4))
    plt.hist(err, bins=40, alpha=0.9)
    plt.xlabel("||error|| [m]"); plt.title(title); plt.tight_layout()
    savefig(path)

def eval_model_on_suffixes_gbr(gbr_predictor, X_builder, suffixes, feature_mode, tag, out_subdir):
    all_rows=[]
    for sfx in suffixes:
        sel = select_triplets_for_suffix(sfx)
        if sel.empty:
            print(f"[test {sfx}] no selected triplets")
            continue
        X_raw, y, feat_cols = X_builder(sel, feature_mode)
        Xs = gbr_predictor["prep"](X_raw)  # preprocessed X
        y_pred = gbr_predictor["pred"](Xs) # np array [N,2]

        df = pd.DataFrame({
            "suffix": sfx,
            "x_true": y[:,0], "y_true": y[:,1],
            "x_pred": y_pred[:,0], "y_pred": y_pred[:,1],
            "tbin": sel["tbin"].values, "cluster": sel["cluster"].values
        })
        all_rows.append(df)

        # save predictions CSV per suffix
        df.to_csv(os.path.join(out_subdir, f"{tag}_{feature_mode}_GBR_{sfx}_predictions.csv"), index=False)

        # plots: scatter + hist
        scatter_plot(y, y_pred,
                     f"Test {sfx} — GBR ({feature_mode})",
                     os.path.join(out_subdir, f"{tag}_{feature_mode}_GBR_{sfx}_test_scatter.png"))
        hist_plot(y, y_pred,
                  f"Test {sfx} — GBR ({feature_mode}) error",
                  os.path.join(out_subdir, f"{tag}_{feature_mode}_GBR_{sfx}_test_errhist.png"))

    return pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()

# =============== Helpers for CSM tuning ===============

def build_train_sel_for_current_csm(train_suffixes: List[str]) -> pd.DataFrame:
    """
    Build concatenated selected triplets for current global CSM configuration.
    """
    sel_rows=[]
    for sfx in train_suffixes:
        sel = select_triplets_for_suffix(sfx)
        if sel.empty:
            print(f"[CSM] {sfx}: 0 rows (current geometry)")
            continue
        sel_rows.append(sel)
        print(f"[CSM] {sfx}: {len(sel)} rows (current geometry)")
    return pd.concat(sel_rows, ignore_index=True) if sel_rows else pd.DataFrame()

def tune_csm_geometry(train_suffixes: List[str], tag_base: str):
    """
    Coarse grid search over CSM geometry parameters:
      - DBSCAN_EPS
      - DBSCAN_MINPTS
      - SPREAD_MAX_M
      - TIME_TOLERANCE_S

    Scoring:
      - Build triplets for all TRAIN_SUFFIXES.
      - Use feature_mode="angle".
      - Train a fixed GBR baseline.
      - Metric: XY-RMSE on validation.
    """
    # Must be declared BEFORE any assignment
    global DBSCAN_EPS, DBSCAN_MINPTS, SPREAD_MAX_M, TIME_TOLERANCE_S

    ensure_dir(OUT_DIR)
    out_subdir = os.path.join(OUT_DIR, f"{tag_base}_CSM_geometry_tuning")
    ensure_dir(out_subdir)

    geometry_grid = [
        {"eps": 0.15, "minpts": 3, "spread": 0.25, "time_tol": 0.30},
        {"eps": 0.20, "minpts": 3, "spread": 0.25, "time_tol": 0.30},     
        {"eps": 0.25, "minpts": 3, "spread": 0.25, "time_tol": 0.30},
        {"eps": 0.30, "minpts": 3, "spread": 0.30, "time_tol": 0.35},
        {"eps": 0.35, "minpts": 3, "spread": 0.35, "time_tol": 0.40},
        {"eps": 0.30, "minpts": 4, "spread": 0.30, "time_tol": 0.35},
        {"eps": 0.35, "minpts": 4, "spread": 0.35, "time_tol": 0.40},
    ]

    records=[]
    best_cfg=None
    best_rmse=float("inf")

    for i, cfg in enumerate(geometry_grid):
        print(f"\n[CSM tune] geometry config {i+1}/{len(geometry_grid)}: {cfg}")

        # Update global knobs
        DBSCAN_EPS       = cfg["eps"]
        DBSCAN_MINPTS    = cfg["minpts"]
        SPREAD_MAX_M     = cfg["spread"]
        TIME_TOLERANCE_S = cfg["time_tol"]

        train_sel = build_train_sel_for_current_csm(train_suffixes)
        n_rows = len(train_sel)

        if n_rows < 200:
            print(f"  [skip] only {n_rows} triplets; skipping config.")
            records.append({**cfg, "n_triplets": n_rows, "val_xy_rmse": np.nan})
            continue

        X_raw, y, _ = build_Xy_from_triplets(train_sel, "angle")
        Xtr_raw, Xval_raw, ytr, yval = train_test_split(
            X_raw, y, test_size=VAL_SPLIT, random_state=SEED
        )

        imp = SimpleImputer(strategy="mean")
        sca = StandardScaler()
        Xtr = sca.fit_transform(imp.fit_transform(Xtr_raw))
        Xval = sca.transform(imp.transform(Xval_raw))

        gbr_base = GradientBoostingRegressor(
            n_estimators=300,
            learning_rate=0.08,
            max_depth=3,
            subsample=0.9,
            loss="quantile",
            random_state=SEED
        )
        gbr = MultiOutputRegressor(gbr_base)
        gbr.fit(Xtr, ytr)

        yval_pred = gbr.predict(Xval)
        df_va = pd.DataFrame({
            "x_true": yval[:,0], "y_true": yval[:,1],
            "x_pred": yval_pred[:,0], "y_pred": yval_pred[:,1]
        })
        m_va = metrics_from_df(df_va)
        val_rmse = m_va["xy_rmse"]

        print(f"  [CSM tune] val XY-RMSE = {val_rmse:.3f} m (n_triplets={n_rows})")

        records.append({**cfg, "n_triplets": n_rows, "val_xy_rmse": val_rmse})

        if np.isfinite(val_rmse) and val_rmse < best_rmse:
            best_rmse = val_rmse
            best_cfg = cfg.copy()

    df_geom = pd.DataFrame(records)
    df_geom.to_csv(os.path.join(out_subdir, f"{tag_base}_CSM_geometry_grid.csv"), index=False)

    if best_cfg is None:
        print("[CSM tune] No valid configuration found. Using defaults.")
        return None

    # Set best globals
    DBSCAN_EPS       = best_cfg["eps"]
    DBSCAN_MINPTS    = best_cfg["minpts"]
    SPREAD_MAX_M     = best_cfg["spread"]
    TIME_TOLERANCE_S = best_cfg["time_tol"]

    with open(os.path.join(out_subdir, f"{tag_base}_CSM_geometry_best.txt"), "w") as f:
        f.write("Best CSM geometry configuration:\n")
        for k, v in best_cfg.items():
            f.write(f"{k}: {v}\n")
        f.write(f"\nVal XY-RMSE: {best_rmse:.6f} m\n")

    print(f"[CSM tune] Best config = {best_cfg}  (XY-RMSE={best_rmse:.3f} m)")
    return best_cfg

# =============== GBR hyperparameter optimisation ===============

def tune_gbr_hyperparams(Xtr_raw: np.ndarray,
                         ytr: np.ndarray,
                         Xval_raw: np.ndarray,
                         yval: np.ndarray,
                         tag_base: str,
                         feature_mode: str,
                         out_subdir: str) -> dict:
    """
    Simple manual grid search for GBR hyperparameters using train/val split.
    Selection metric: XY-RMSE on validation.
    Returns a dict with:
        "params"         - best hyperparameter dict
        "imp"            - fitted imputer
        "sca"            - fitted scaler
        "model"          - fitted MultiOutputRegressor(GBR)
        "train_metrics"  - metrics on train
        "val_metrics"    - metrics on val
    Also saves a CSV with all tried configs + their metrics.
    """
    # Grid can be extended if you want more exhaustive search
    param_grid = [
        {"n_estimators": 200, "learning_rate": 0.05, "max_depth": 2, "subsample": 0.8},
        {"n_estimators": 300, "learning_rate": 0.05, "max_depth": 3, "subsample": 0.8},
        {"n_estimators": 400, "learning_rate": 0.10, "max_depth": 3, "subsample": 0.9},
        {"n_estimators": 600, "learning_rate": 0.05, "max_depth": 4, "subsample": 0.9},
        {"n_estimators": 600, "learning_rate": 0.1, "max_depth": 3, "subsample": 0.9},
        {"n_estimators": 600, "learning_rate": 0.15, "max_depth": 4, "subsample": 0.9},
        {"n_estimators": 800, "learning_rate": 0.05, "max_depth": 1, "subsample": 0.9},
        {"n_estimators": 800, "learning_rate": 0.1, "max_depth": 3, "subsample": 0.9},
        {"n_estimators": 1000, "learning_rate": 0.1, "max_depth": 3, "subsample": 0.9},
        {"n_estimators": 1000, "learning_rate": 0.05, "max_depth": 4, "subsample": 0.9},
        {"n_estimators": 1500, "learning_rate": 0.05, "max_depth": 4, "subsample": 0.9},
        {"n_estimators": 1500, "learning_rate": 0.1, "max_depth": 4, "subsample": 0.9},
        {"n_estimators": 2000, "learning_rate": 0.1, "max_depth": 4, "subsample": 0.9},
        {"n_estimators": 2500, "learning_rate": 0.1, "max_depth": 4, "subsample": 0.9},
        {"n_estimators": 3000, "learning_rate": 0.1, "max_depth": 4, "subsample": 0.9},


    ]

    results = []
    best_cfg = None
    best_val_rmse = float("inf")

    for i, params in enumerate(param_grid):
        print(f"[GBR tune {feature_mode}] config {i+1}/{len(param_grid)}: {params}")

        imp = SimpleImputer(strategy="mean")
        sca = StandardScaler()

        Xtr0 = imp.fit_transform(Xtr_raw)
        Xval0 = imp.transform(Xval_raw)
        Xtr = sca.fit_transform(Xtr0)
        Xval = sca.transform(Xval0)

        gbr_base = GradientBoostingRegressor(
            n_estimators=params["n_estimators"],
            learning_rate=params["learning_rate"],
            max_depth=params["max_depth"],
            subsample=params["subsample"],
            loss="squared_error",
            random_state=SEED
        )
        gbr = MultiOutputRegressor(gbr_base)
        gbr.fit(Xtr, ytr)

        # Train metrics
        ytr_pred = gbr.predict(Xtr)
        df_tr = pd.DataFrame({
            "x_true": ytr[:,0], "y_true": ytr[:,1],
            "x_pred": ytr_pred[:,0], "y_pred": ytr_pred[:,1]
        })
        m_tr = metrics_from_df(df_tr)

        # Val metrics
        yval_pred = gbr.predict(Xval)
        df_va = pd.DataFrame({
            "x_true": yval[:,0], "y_true": yval[:,1],
            "x_pred": yval_pred[:,0], "y_pred": yval_pred[:,1]
        })
        m_va = metrics_from_df(df_va)

        row = {
            "n_estimators": params["n_estimators"],
            "learning_rate": params["learning_rate"],
            "max_depth": params["max_depth"],
            "subsample": params["subsample"],
            "train_xy_rmse": m_tr["xy_rmse"],
            "train_xy_mae":  m_tr["xy_mae"],
            "val_xy_rmse":   m_va["xy_rmse"],
            "val_xy_mae":    m_va["xy_mae"],
        }
        results.append(row)

        if m_va["xy_rmse"] < best_val_rmse:
            best_val_rmse = m_va["xy_rmse"]
            best_cfg = {
                "params": params,
                "imp": imp,
                "sca": sca,
                "model": gbr,
                "train_metrics": m_tr,
                "val_metrics": m_va,
            }

    # Save grid search results
    df_grid = pd.DataFrame(results)
    df_grid.to_csv(os.path.join(
        out_subdir,
        f"{tag_base}_{feature_mode}_GBR_hyperparam_grid.csv"
    ), index=False)

    print(f"[GBR tune {feature_mode}] best val XY-RMSE={best_val_rmse:.3f} with params={best_cfg['params']}")

    # Save best params as a small text file
    best_txt_path = os.path.join(
        out_subdir,
        f"{tag_base}_{feature_mode}_GBR_best_params.txt"
    )
    with open(best_txt_path, "w") as f:
        f.write(f"Best params (feature_mode={feature_mode}):\n")
        for k,v in best_cfg["params"].items():
            f.write(f"{k}: {v}\n")
        f.write(f"\nBest val XY-RMSE: {best_val_rmse:.6f}\n")

    return best_cfg

# =============== Runner for one feature_mode ===============

def run_gbr_for_feature_mode(train_sel: pd.DataFrame, feature_mode: str, tag_base: str):
    """
    Train/eval GBR (squared error) for one feature_mode.
    Steps:
      - Build X,y from selected triplets.
      - Train/val split.
      - Hyperparameter tuning (manual grid search) using val XY-RMSE.
      - Save:
          * hyperparameter grid results
          * best-GBR train/val metrics CSV + bar plot
          * best hyperparameters (TXT)
          * test metrics CSV (all + cc2-only)
          * predictions & plots for TEST_SUFFIXES
    """
    ensure_dir(OUT_DIR)
    out_subdir = os.path.join(OUT_DIR, f"{tag_base}_{feature_mode}_GBR")
    ensure_dir(out_subdir)

    # === Build X,y
    X_raw, y, feat_cols = build_Xy_from_triplets(train_sel, feature_mode)
    if len(X_raw)==0:
        raise RuntimeError(f"No training rows for feature_mode={feature_mode}")

    # shared train/val split
    Xtr_raw, Xval_raw, ytr, yval = train_test_split(X_raw, y, test_size=VAL_SPLIT, random_state=SEED)

    # === Hyperparameter optimisation for GBR
    best_cfg = tune_gbr_hyperparams(
        Xtr_raw, ytr, Xval_raw, yval,
        tag_base=tag_base,
        feature_mode=feature_mode,
        out_subdir=out_subdir
    )

    imp_gb = best_cfg["imp"]
    sca_gb = best_cfg["sca"]
    gbr     = best_cfg["model"]
    m_tr    = best_cfg["train_metrics"]
    m_va    = best_cfg["val_metrics"]

    def gbr_prep(X):
        X0 = imp_gb.transform(X)
        return sca_gb.transform(X0)

    gbr_predictor = {
        "name": "GBR",
        "prep": gbr_prep,
        "pred": lambda Xs: gbr.predict(Xs)
    }

    # === Train/val metrics (from best config)
    ytr_pred = gbr_predictor["pred"](gbr_predictor["prep"](Xtr_raw))
    yval_pred = gbr_predictor["pred"](gbr_predictor["prep"](Xval_raw))

    df_tr = pd.DataFrame({
        "x_true": ytr[:,0], "y_true": ytr[:,1],
        "x_pred": ytr_pred[:,0], "y_pred": ytr_pred[:,1]
    })
    df_va = pd.DataFrame({
        "x_true": yval[:,0], "y_true": yval[:,1],
        "x_pred": yval_pred[:,0], "y_pred": yval_pred[:,1]
    })

    # re-compute metrics for consistency (or use m_tr/m_va directly)
    m_tr = metrics_from_df(df_tr)
    m_va = metrics_from_df(df_va)

    df_tv = pd.DataFrame([
        {"subset": "train", **m_tr},
        {"subset": "val",   **m_va}
    ])
    df_tv.to_csv(os.path.join(out_subdir,
        f"{tag_base}_{feature_mode}_GBR_train_val_metrics.csv"), index=False)

    # bar plot: train vs val XY RMSE & MAE
    rmse_vals = [m_tr["xy_rmse"], m_va["xy_rmse"]]
    mae_vals  = [m_tr["xy_mae"],  m_va["xy_mae"]]
    labels    = ["train", "val"]

    x = np.arange(len(labels))
    width = 0.35

    plt.figure(figsize=(6,4))
    plt.bar(x - width/2, rmse_vals, width, label="XY RMSE")
    plt.bar(x + width/2, mae_vals,  width, label="XY MAE")
    plt.xticks(x, labels)
    plt.ylabel("Error [m]")
    plt.title(f"{tag_base} ({feature_mode}) — GBR train/val metrics (tuned)")
    plt.legend()
    plt.tight_layout()
    savefig(os.path.join(out_subdir,
        f"{tag_base}_{feature_mode}_GBR_train_val_bar.png"))

    # === Evaluate on TEST suffixes with the tuned model
    df_test = eval_model_on_suffixes_gbr(gbr_predictor, build_Xy_from_triplets,
                                         TEST_SUFFIXES, feature_mode,
                                         tag=tag_base, out_subdir=out_subdir)

    metrics_rows=[]
    if not df_test.empty:
        m_all = metrics_from_df(df_test)
        metrics_rows.append({"feature_mode":feature_mode, "model":"GBR", **m_all})
        df_cc2 = df_test[df_test["suffix"]=="cc2"]
        if not df_cc2.empty:
            m_cc2 = metrics_from_df(df_cc2)
            metrics_rows.append({"feature_mode":feature_mode, "model":"GBR_cc2_only", **m_cc2})

    if metrics_rows:
        df_metrics = pd.DataFrame(metrics_rows)
        df_metrics.to_csv(os.path.join(out_subdir,
            f"{tag_base}_{feature_mode}_GBR_test_metrics.csv"), index=False)

# =============== Main ===============

def main():
    set_seed(SEED)
    ensure_dir(OUT_DIR)
    print(f"[config] device={DEVICE}")
    print("[info] Triplet selector = min-dmid (geometric heuristic)")
    print("[info] Localization model = GBR (squared_error) with hyperparameter tuning")

    # 0) Tune CSM geometry (DBSCAN + gates) using angle-mode + GBR baseline
    print("\n[step] Tuning CSM geometry (DBSCAN_EPS, MINPTS, SPREAD_MAX_M, TIME_TOLERANCE_S)...")
    best_geom = tune_csm_geometry(TRAIN_SUFFIXES, tag_base="CSM")
    if best_geom is not None:
        print(f"[info] Using tuned CSM geometry: {best_geom}")
    else:
        print("[info] Using default CSM geometry (tuning failed or skipped).")

    # 1) Build selected triplets for TRAIN using final CSM geometry
    print("\n[step] Building train-selected triplets with final CSM geometry...")
    sel_rows=[]
    for sfx in TRAIN_SUFFIXES:
        sel = select_triplets_for_suffix(sfx)
        if sel.empty:
            print(f"[train sel] {sfx}: 0 rows"); continue
        sel_rows.append(sel)
        print(f"[train sel] {sfx}: {len(sel)} rows")
    if not sel_rows:
        print("[abort] no selected triplets on TRAIN."); return
    train_sel = pd.concat(sel_rows, ignore_index=True)
    train_sel.to_csv(os.path.join(OUT_DIR, "train_selected_triplets_min_dmid.csv"), index=False)

    # 2) Run GBR for BOTH feature modes (angle & range)
    for FEATURE_MODE in ["angle","range"]:
        print(f"\n=== Running GBR for FEATURE_MODE={FEATURE_MODE} ===")
        run_gbr_for_feature_mode(train_sel, FEATURE_MODE, tag_base="CSM")

if __name__=="__main__":
    main()


[config] device=cuda
[info] Triplet selector = min-dmid (geometric heuristic)
[info] Localization model = GBR (squared_error) with hyperparameter tuning

[step] Tuning CSM geometry (DBSCAN_EPS, MINPTS, SPREAD_MAX_M, TIME_TOLERANCE_S)...

[CSM tune] geometry config 1/7: {'eps': 0.15, 'minpts': 3, 'spread': 0.25, 'time_tol': 0.3}
[CSM] rr1: 44 rows (current geometry)
[CSM] rr2: 30 rows (current geometry)
[CSM] rr3: 49 rows (current geometry)
[CSM] rr4: 35 rows (current geometry)
[CSM] rr5: 38 rows (current geometry)
[CSM] rr6: 35 rows (current geometry)
[CSM] rr7: 29 rows (current geometry)
[CSM] rr8: 49 rows (current geometry)
[CSM] rr9: 54 rows (current geometry)
[CSM] rr12: 40 rows (current geometry)
[CSM] rr11: 40 rows (current geometry)
[CSM] cc3: 35 rows (current geometry)
[CSM] cc5: 47 rows (current geometry)
[CSM] cc6: 38 rows (current geometry)
[CSM] cc7: 28 rows (current geometry)
[CSM] cc8: 39 rows (current geometry)
[CSM] pcc3: 85 rows (current geometry)
[CSM] prr1: 38 rows (